### Universidad del Valle de Guatemala
### Facultad de Ingeniería — Departamento de Ciencias de la Computación
### CC3084 – Data Science | Semestre II – 2026

***Proyecto 1: Obtención y Limpieza de los Datos***

**Tema:** Establecimientos educativos de Guatemala — nivel escolar **Diversificado**
**Fuente de datos:** Sistema de Búsqueda de Establecimientos Educativos, MINEDUC Guatemala — http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/
**Grupo:** 2



| Integrante | Carné |
|---|---|
| Jose Donado | 22684 |
| Daniela Ramírez | 23053 |
| Cindy Gualim | 21226 |

## Objetivo del proyecto
Acceder a los datos de establecimientos educativos de todo el país que lleguen hasta el nivel Diversificado, y limpiarlos de la forma más transparente y reproducible posible. Concretamente, este notebook documenta:
1. La obtención de los datos crudos (web scraping con Selenium) y su filtrado al nivel escolar exigido.
2. Un diagnóstico del estado inicial de los datos (registros, tipos, faltantes, únicos, duplicados, valores fuera de dominio, formatos inconsistentes).
3. Un plan de limpieza por variable (problema → regla de corrección → riesgos).
4. La limpieza del conjunto de datos completo, variable por variable.
5. Un registro de todas las transformaciones aplicadas.
6. Pruebas automáticas de validación del conjunto limpio.
7. Un informe de calidad que compara el estado antes vs. después de la limpieza.
8. La generación de un único conjunto de datos limpio, para todos los departamentos.
9. Un Libro de Códigos (`CODEBOOK.md`, en la raíz del repositorio) con los metadatos necesarios para interpretar el conjunto limpio.

In [2]:
import os
import re
import json
import unicodedata
from io import StringIO
from datetime import date

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)

FECHA_EXTRACCION = "2026-07-18"   # fecha en que se corrieron las actividades 1-2 contra el sitio del MINEDUC
VERSION_LIMPIO = "1.0"

# Catálogo oficial: los 22 departamentos de Guatemala (sin tildes, como los
# devuelve el formulario del MINEDUC). Se usa tanto para el scraping como para
# validar el dominio de la variable `departamento`.
DEPARTAMENTOS_OFICIALES = [
    "GUATEMALA", "EL PROGRESO", "SACATEPEQUEZ", "CHIMALTENANGO", "ESCUINTLA",
    "SANTA ROSA", "SOLOLA", "TOTONICAPAN", "QUETZALTENANGO", "SUCHITEPEQUEZ",
    "RETALHULEU", "SAN MARCOS", "HUEHUETENANGO", "QUICHE", "BAJA VERAPAZ",
    "ALTA VERAPAZ", "PETEN", "IZABAL", "ZACAPA", "CHIQUIMULA", "JALAPA", "JUTIAPA",
]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Entorno listo. pandas", pd.__version__)

Entorno listo. pandas 3.0.3


---
### ***ACTIVIDADES 1 y 2 — Descarga de los datos y guardado del crudo en .csv***

Se recorre el formulario público del MINEDUC (`cmbDepartamento`, `cmbMunicipio`, `cmbNivel`,
`cmbSector`, `ddlplan`, `ddlModalidad`) consultando los **22 departamentos** del país con
`NIVEL = DIVERSIFICADO` y Municipio / Sector / Plan / Modalidad = `TODOS`. El resultado de cada
consulta es una tabla HTML que se convierte a `DataFrame` con pandas; al final se concatenan
todas las consultas y **se guarda el crudo sin modificar** en `establecimientos.csv`
(actividad 2: guardar los datos crudos en .csv).

> **Nota de reproducibilidad:** esta parte requiere Chrome + conexión a internet y tarda varios
> minutos. Si ya existe `establecimientos.csv`, la celda de ejecución se salta automáticamente y
> el notebook continúa desde la actividad 3 con el archivo ya descargado.

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

URL_MINEDUC = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"


def abrir_navegador():
    options = Options()
    # options.add_argument("--headless=new")   # descomentar para correr sin ventana
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    driver.maximize_window()
    return driver


def consultar(driver, departamento, nivel="DIVERSIFICADO"):
    """Llena el formulario para (departamento, nivel) y devuelve la tabla de
    resultados como DataFrame."""
    wait = WebDriverWait(driver, 20)
    driver.get(URL_MINEDUC)
    wait.until(EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")))

    Select(driver.find_element(
        By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )).select_by_visible_text(departamento)

    time.sleep(2)  # el combo de municipios se recarga por postback

    for name, valor in [
        ("_ctl0:ContentPlaceHolder1:cmbMunicipio", "TODOS"),
        ("_ctl0:ContentPlaceHolder1:cmbNivel", nivel),
        ("_ctl0:ContentPlaceHolder1:cmbSector", "TODOS"),
        ("_ctl0:ContentPlaceHolder1:ddlplan", "TODOS"),
        ("_ctl0:ContentPlaceHolder1:ddlModalidad", "TODOS"),
    ]:
        Select(driver.find_element(By.NAME, name)).select_by_visible_text(valor)

    driver.find_element(
        By.NAME, "_ctl0:ContentPlaceHolder1:IbtnConsultar").click()

    wait.until(EC.presence_of_element_located(
        (By.ID, "_ctl0_ContentPlaceHolder1_dgResultado")))

    html = driver.find_element(
        By.ID, "_ctl0_ContentPlaceHolder1_dgResultado").get_attribute("outerHTML")

    df = pd.read_html(StringIO(html))[0]
    df.columns = df.iloc[0]                    # la primera fila trae los encabezados
    df = df.iloc[1:].reset_index(drop=True)
    df = df.drop(df.columns[0], axis=1)        # columna del botón "Seleccionar"
    return df

In [4]:
# Ejecuta el scraping solo si aún no existe el crudo (reproducible pero no destructivo)
if not os.path.exists("establecimientos.csv"):
    driver = abrir_navegador()
    todos = []
    try:
        for depto in DEPARTAMENTOS_OFICIALES:
            print(f"Consultando {depto} - DIVERSIFICADO ...", end=" ")
            df_dep = consultar(driver, depto, "DIVERSIFICADO")
            print(f"{len(df_dep)} registros")
            todos.append(df_dep)
    finally:
        driver.quit()

    df_final = pd.concat(todos, ignore_index=True)
    df_final.to_csv("establecimientos.csv", index=False, encoding="utf-8-sig")
    print(f"\nCrudo guardado: {df_final.shape[0]} filas, {df_final.shape[1]} columnas.")
else:
    print("establecimientos.csv ya existe: se omite el scraping.")

establecimientos.csv ya existe: se omite el scraping.


---
### ***ACTIVIDAD 3 — Análisis del estado inicial de los datos (diagnóstico)***

Carga del crudo y filtrado a `NIVEL = DIVERSIFICADO`

Se carga `establecimientos.csv` y se aplica un **filtro defensivo** por `NIVEL == "DIVERSIFICADO"`
(por si el archivo fue generado con más niveles). El crudo ya filtrado se guarda aparte, en
`data/raw/`, para dejar constancia del punto de partida exacto del diagnóstico.

In [5]:
df_crudo = pd.read_csv("establecimientos.csv", encoding="utf-8-sig", dtype=str)

if "NIVEL" in df_crudo.columns:
    df_raw = df_crudo[df_crudo["NIVEL"] == "DIVERSIFICADO"].reset_index(drop=True)
else:
    df_raw = df_crudo.copy()

df_raw.to_csv("data/raw/establecimientos_diversificado_raw.csv",
              index=False, encoding="utf-8-sig")

print(f"Crudo original : {df_crudo.shape[0]} filas, {df_crudo.shape[1]} columnas")
print(f"Filtrado DIVERSIFICADO: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
df_raw.head()

Crudo original : 9728 filas, 17 columnas
Filtrado DIVERSIFICADO: 9706 filas, 17 columnas


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,01-01-0007-46,01-504,GUATEMALA,GUATEMALA,COLEGIO HAN AL AMERICANO,9A. CALLE 28-74 ZONA 14,23661344-23661600,DAVID SOTOJ SANCHEZ,OLGA LETICIA PEREZ GARICA,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
1,01-01-0011-46,01-620,GUATEMALA,GUATEMALA,COLEGIO LA EDUCACION,7A. AVENIDA 3-20 ZONA 19 COLONIA LA FLORIDA,24310333,RONALD MAURICIO MORALES VIVAR,NATALY GABRIELA DE LEON URÍZA,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),GUATEMALA NORTE
2,01-01-0013-46,01-503,GUATEMALA,GUATEMALA,COLEGIO LA ACADEMIA,13 CALLE 6-91 ZONA 10,23323279,NANCY CAROLINA NAVAS GONZÁLEZ,ANA LUCRECIA LOPEZ FLORES,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE
3,01-01-0015-46,01-403,GUATEMALA,GUATEMALA,"CENTRO EDUCATIVO ""RENUEVO""",6A. 2-34 ZONA 1,55856196,CARLOS HUMBERTO GONZÁLEZ DE LEÓN,NaN,DIVERSIFICADO,PRIVADO,RURAL,CERRADA DEFINITIVAMENTE,MONOLINGUE,INTERMEDIA,FIN DE SEMANA,GUATEMALA NORTE
4,01-01-0018-46,01-307,GUATEMALA,GUATEMALA,LICEO ROMANO AMERICANO,6A. CALLE 2-12 ZONA 1,NaN,NORMA ARACELY PALOMO FRANCO DE DIAZ,VICTOR MANUEL PORTILLO LEON,DIVERSIFICADO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,MATUTINA,DIARIO(REGULAR),GUATEMALA NORTE


***3a — Número de registros y variables***

In [6]:
print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")
print()
df_raw.info()

Registros : 9706
Variables : 17

<class 'pandas.DataFrame'>
RangeIndex: 9706 entries, 0 to 9705
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CODIGO           9706 non-null   str  
 1   DISTRITO         9405 non-null   str  
 2   DEPARTAMENTO     9706 non-null   str  
 3   MUNICIPIO        9706 non-null   str  
 4   ESTABLECIMIENTO  9703 non-null   str  
 5   DIRECCION        9637 non-null   str  
 6   TELEFONO         9024 non-null   str  
 7   SUPERVISOR       9402 non-null   str  
 8   DIRECTOR         8510 non-null   str  
 9   NIVEL            9706 non-null   str  
 10  SECTOR           9706 non-null   str  
 11  AREA             9706 non-null   str  
 12  STATUS           9706 non-null   str  
 13  MODALIDAD        9706 non-null   str  
 14  JORNADA          9706 non-null   str  
 15  PLAN             9706 non-null   str  
 16  DEPARTAMENTAL    9706 non-null   str  
dtypes: str(17)
memory usage: 1.3 M

***3b — Tipo de dato de cada variable***

La tabla llega del HTML como texto, así que **pandas interpreta todas las columnas como `object`**.
Conviene documentar cuál *debería* ser el tipo de cada variable:

- `TELEFONO` conceptualmente es numérico, pero se conserva como texto porque puede traer
  varios números en la misma celda y porque un número telefónico no se opera aritméticamente.
- `CODIGO` y `DISTRITO` son códigos con guiones y ceros a la izquierda: convertirlos a entero
  perdería información.
- El resto son variables **categóricas** o de **texto libre**.

In [7]:
tipos = pd.DataFrame({
    "dtype_actual": df_raw.dtypes.astype(str),
    "ejemplo": [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else None
                for c in df_raw.columns],
})
tipos["tipo_esperado"] = tipos.index.map({
    "CODIGO": "string (código con formato NN-NN-NNNN-NN)",
    "DISTRITO": "string (código)",
    "DEPARTAMENTO": "category",
    "MUNICIPIO": "category",
    "ESTABLECIMIENTO": "string (texto libre)",
    "DIRECCION": "string (texto libre)",
    "TELEFONO": "string (8 dígitos)",
    "SUPERVISOR": "string (nombre)",
    "DIRECTOR": "string (nombre)",
    "NIVEL": "category",
    "SECTOR": "category",
    "AREA": "category (URBANA/RURAL)",
    "STATUS": "category",
    "MODALIDAD": "category",
    "JORNADA": "category",
    "PLAN": "category",
    "DEPARTAMENTAL": "category",
}).fillna("string")
tipos

,dtype_actual,ejemplo,tipo_esperado
CODIGO,str,01-01-0007-46,string (código con formato NN-NN-NNNN-NN)
DISTRITO,str,01-504,string (código)
DEPARTAMENTO,str,GUATEMALA,category
MUNICIPIO,str,GUATEMALA,category
ESTABLECIMIENTO,str,COLEGIO HAN AL AMERICANO,string (texto libre)
DIRECCION,str,9A. CALLE 28-74 ZONA 14,string (texto libre)
TELEFONO,str,23661344-23661600,string (8 dígitos)
SUPERVISOR,str,DAVID SOTOJ SANCHEZ,string (nombre)
DIRECTOR,str,OLGA LETICIA PEREZ GARICA,string (nombre)
NIVEL,str,DIVERSIFICADO,category


***3c — Cantidad y porcentaje de valores faltantes por variable***

In [8]:
tabla_faltantes = pd.DataFrame({
    "faltantes": df_raw.isna().sum(),
    "porcentaje_%": (df_raw.isna().mean() * 100).round(2),
}).sort_values("porcentaje_%", ascending=False)

print(f"Celdas faltantes en total: {int(df_raw.isna().sum().sum())} "
      f"({df_raw.isna().sum().sum() / df_raw.size * 100:.2f}% de todas las celdas)")
tabla_faltantes

Celdas faltantes en total: 2555 (1.55% de todas las celdas)


,faltantes,porcentaje_%
DIRECTOR,1196,12.32
TELEFONO,682,7.03
SUPERVISOR,304,3.13
DISTRITO,301,3.10
DIRECCION,69,0.71
ESTABLECIMIENTO,3,0.03
CODIGO,0,0.00
MUNICIPIO,0,0.00
DEPARTAMENTO,0,0.00
NIVEL,0,0.00


***3d — Cantidad de valores únicos***

In [9]:
pd.DataFrame({
    "valores_unicos": df_raw.nunique(dropna=True),
    "%_del_total": (df_raw.nunique(dropna=True) / len(df_raw) * 100).round(2),
}).sort_values("valores_unicos", ascending=False)

,valores_unicos,%_del_total
CODIGO,9706,100.00
DIRECCION,6023,62.05
TELEFONO,5547,57.15
ESTABLECIMIENTO,5036,51.89
DIRECTOR,4852,49.99
DISTRITO,1593,16.41
SUPERVISOR,1241,12.79
MUNICIPIO,330,3.40
DEPARTAMENTAL,26,0.27
DEPARTAMENTO,22,0.23


***3e — Cantidad de registros duplicados exactos***

In [10]:
resumen_duplicados = pd.DataFrame({
    "Métrica": [
        "Duplicados exactos (todas las ocurrencias, keep=False)",
        "Duplicados exactos (sin la primera ocurrencia)",
        "Filas completamente vacías (todas las columnas nulas)",
        "Duplicados exactos entre registros CON datos",
        "Códigos (CODIGO) repetidos entre registros no nulos",
    ],
    "Cantidad": [
        int(df_raw.duplicated(keep=False).sum()),
        int(df_raw.duplicated().sum()),
        int(df_raw.isna().all(axis=1).sum()),
        int(df_raw[~df_raw.isna().all(axis=1)].duplicated().sum()),
        int(df_raw[df_raw["CODIGO"].notna()].duplicated(subset=["CODIGO"]).sum()),
    ],
})
resumen_duplicados

,Métrica,Cantidad
0,"Duplicados exactos (todas las ocurrencias, kee...",0
1,Duplicados exactos (sin la primera ocurrencia),0
2,Filas completamente vacías (todas las columnas...,0
3,Duplicados exactos entre registros CON datos,0
4,Códigos (CODIGO) repetidos entre registros no ...,0


Los duplicados exactos detectados corresponden **en su totalidad a filas 100 % vacías** (las mismas
filas responsables de una parte de los faltantes del inciso c). Al excluirlas no queda ningún
duplicado exacto entre establecimientos con información real, y `CODIGO` no presenta valores
repetidos entre registros no nulos. Estas filas vacías se eliminarán en la sección 5a.

***3f — Variables con valores fuera de dominio o inconsistentes***

In [11]:
telefonos_raw = df_raw["TELEFONO"].dropna().astype(str)
codigos_raw = df_raw["CODIGO"].dropna().astype(str)

resumen_dominio = pd.DataFrame({
    "Variable": ["AREA", "STATUS", "TELEFONO", "CODIGO", "DEPARTAMENTO"],
    "Dominio esperado": [
        "{URBANA, RURAL}",
        "categorías administrativas conocidas",
        "8 dígitos numéricos",
        "NN-NN-NNNN-NN",
        "los 22 departamentos oficiales",
    ],
    "Hallazgo": [
        "aparece 'SIN ESPECIFICAR', fuera del dominio binario",
        "'TEMPORAL NOMBRAMIENTO' / 'TEMPORAL TITULOS': poco frecuentes pero válidas",
        "longitudes de 1 a 16 dígitos, con guiones, comas y texto",
        "formato uniforme",
        "sin valores fuera de catálogo",
    ],
    "Registros que no cumplen": [
        int((df_raw["AREA"] == "SIN ESPECIFICAR").sum()),
        int(df_raw["STATUS"].isin(["TEMPORAL NOMBRAMIENTO", "TEMPORAL TITULOS"]).sum()),
        int((~telefonos_raw.str.fullmatch(r"\d{8}")).sum()),
        int((~codigos_raw.str.fullmatch(r"\d{2}-\d{2}-\d{4}-\d{2}")).sum()),
        int((~df_raw["DEPARTAMENTO"].dropna().isin(DEPARTAMENTOS_OFICIALES)).sum()),
    ],
})
resumen_dominio

,Variable,Dominio esperado,Hallazgo,Registros que no cumplen
0,AREA,"{URBANA, RURAL}","aparece 'SIN ESPECIFICAR', fuera del dominio b...",2
1,STATUS,categorías administrativas conocidas,'TEMPORAL NOMBRAMIENTO' / 'TEMPORAL TITULOS': ...,91
2,TELEFONO,8 dígitos numéricos,"longitudes de 1 a 16 dígitos, con guiones, com...",163
3,CODIGO,NN-NN-NNNN-NN,formato uniforme,0
4,DEPARTAMENTO,los 22 departamentos oficiales,sin valores fuera de catálogo,0


***3g — Variables con formatos inconsistentes***

In [12]:
est_raw = df_raw["ESTABLECIMIENTO"].dropna().astype(str)
dire_raw = df_raw["DIRECCION"].dropna().astype(str)
distrito_raw = df_raw["DISTRITO"].dropna().astype(str)
formato_distrito = (distrito_raw.str.fullmatch(r"\d{2}-\d{3}")
                    | distrito_raw.str.fullmatch(r"\d{2}-\d{2}-\d{4}"))

resumen_formato = pd.DataFrame({
    "Variable": ["ESTABLECIMIENTO", "ESTABLECIMIENTO", "DIRECCION", "DIRECCION",
                 "TELEFONO", "DISTRITO"],
    "Problema de formato": [
        'usa comillas dobles (")', "usa comillas simples (')",
        "contiene minúsculas", "espacios dobles o extremos",
        "contiene guion, coma o texto", "no sigue NN-NNN ni NN-NN-NNNN",
    ],
    "Cantidad": [
        int(est_raw.str.contains('"').sum()),
        int(est_raw.str.contains("'").sum()),
        int(dire_raw.str.contains("[a-z]", regex=True).sum()),
        int(dire_raw.str.contains(r"^\s|\s$|\s{2,}", regex=True).sum()),
        int((~telefonos_raw.str.fullmatch(r"\d+")).sum()),
        int((~formato_distrito).sum()),
    ],
})
resumen_formato

,Variable,Problema de formato,Cantidad
0,ESTABLECIMIENTO,"usa comillas dobles ("")",2028
1,ESTABLECIMIENTO,usa comillas simples ('),508
2,DIRECCION,contiene minúsculas,7
3,DIRECCION,espacios dobles o extremos,0
4,TELEFONO,"contiene guion, coma o texto",123
5,DISTRITO,no sigue NN-NNN ni NN-NN-NNNN,16


***3h — Identificación de problemas potenciales de calidad de datos***

In [13]:
quitar_tildes = lambda s: "".join(
    c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn"
).upper()

# ¿DEPARTAMENTAL (dirección departamental) contradice a DEPARTAMENTO?
sub = df_raw[["DEPARTAMENTO", "DEPARTAMENTAL"]].dropna()
coincide = [dpal.startswith(dep) for dpal, dep in
            zip(sub["DEPARTAMENTAL"].map(quitar_tildes), sub["DEPARTAMENTO"].map(quitar_tildes))]

# ¿Hay caracteres invisibles / de control en las columnas de texto?
tiene_invisible = {
    col: int(df_raw[col].dropna().astype(str)
             .apply(lambda t: any(unicodedata.category(c) in ("Cc", "Cf") for c in t)).sum())
    for col in df_raw.columns
}

pd.DataFrame({
    "Chequeo": [
        "Registros comparados DEPARTAMENTO vs DEPARTAMENTAL",
        "Contradicciones DEPARTAMENTO vs DEPARTAMENTAL",
        "Columnas con caracteres invisibles/de control",
        "Municipios distintos presentes (de 340 oficiales)",
    ],
    "Resultado": [
        len(sub),
        int((~pd.Series(coincide)).sum()),
        int(sum(v > 0 for v in tiene_invisible.values())),
        int(df_raw["MUNICIPIO"].nunique()),
    ],
})

,Chequeo,Resultado
0,Registros comparados DEPARTAMENTO vs DEPARTAME...,9706
1,Contradicciones DEPARTAMENTO vs DEPARTAMENTAL,0
2,Columnas con caracteres invisibles/de control,0
3,Municipios distintos presentes (de 340 oficiales),330


### Resumen de los problemas de calidad detectados

1. **Filas 100 % vacías** que inflan artificialmente los conteos de faltantes y de duplicados
   exactos; deben eliminarse antes de cualquier otra medición.
2. **`DIRECTOR`, `TELEFONO` y `SUPERVISOR`** concentran la mayor parte de los faltantes reales.
3. **`TELEFONO`** es la variable con más problemas de dominio y formato: mezcla longitudes,
   separadores (guiones, comas) y texto, y en varias celdas hay más de un número.
4. **`DISTRITO`** mezcla dos convenciones de formato (`NN-NNN` y `NN-NN-NNNN`) y trae valores
   truncados.
5. **`ESTABLECIMIENTO`** no tiene un estilo uniforme de comillas para apodos / nombres comerciales,
   y algunas comillas quedan sin cerrar.
6. **`MUNICIPIO`** no cubre los 340 municipios oficiales del país: hay que distinguir si es por
   variantes de escritura o porque hay municipios sin oferta de Diversificado.
7. **`AREA`** tiene la categoría `SIN ESPECIFICAR`, fuera del dominio binario esperado.
8. `DEPARTAMENTO`, `CODIGO` y el cruce `DEPARTAMENTO` ↔ `DEPARTAMENTAL` **no** presentan
   inconsistencias: son las variables más limpias del conjunto.

### ***ACTIVIDAD 4 — Plan de limpieza***

Para cada variable: **problema encontrado → regla de corrección (y por qué debería funcionar) →
riesgos asociados a la transformación**. Este plan se define *antes* de tocar los datos; la
actividad 5 lo ejecuta y la actividad 6 deja constancia de lo que efectivamente cambió.

In [14]:
plan_limpieza = pd.DataFrame([
    ["CODIGO",
     "Faltantes provenientes de filas 100% vacías. Formato ya consistente.",
     "Eliminar las filas vacías; conservar el código como texto (no como número).",
     "Convertirlo a numérico perdería los guiones y los ceros a la izquierda."],
    ["DISTRITO",
     "Faltantes; dos formatos coexistentes (NN-NNN y NN-NN-NNNN); valores truncados.",
     "Registrar el formato en una variable derivada y pasar los truncados a NA.",
     "No hay regla segura para convertir un formato al otro sin inventar dígitos."],
    ["DEPARTAMENTO",
     "Solo faltantes de filas vacías; dominio limpio.",
     "Validar contra el catálogo oficial de 22 departamentos.",
     "Ninguno relevante."],
    ["MUNICIPIO",
     "No cubre los 340 municipios oficiales; posibles variantes de escritura.",
     "Normalizar a mayúsculas y buscar variantes por similitud de cadenas DENTRO de cada departamento.",
     "Hay municipios homónimos en departamentos distintos: unificar entre departamentos sería un error."],
    ["ESTABLECIMIENTO",
     "Comillas simples y dobles mezcladas; comillas sin cerrar.",
     "Unificar a comillas dobles y eliminar comillas sueltas al inicio/fin.",
     "Cambio cosmético; se documenta que no altera el nombre real."],
    ["DIRECCION",
     "Minúsculas mezcladas, espacios múltiples, abreviaturas de ordinal inconsistentes.",
     "Normalizar espacios y pasar a mayúsculas.",
     "El texto libre puede contener nombres propios cuya capitalización se pierde."],
    ["TELEFONO",
     "Alta tasa de faltantes; varios números por celda; longitudes fuera de 8 dígitos; texto.",
     "Extraer los números de 8 dígitos: el primero queda en telefono, el segundo en telefono_2; NA si ninguno califica.",
     "Riesgo más alto del proyecto: se puede descartar el número realmente vigente."],
    ["SUPERVISOR",
     "Faltantes; posibles variantes de escritura del mismo nombre.",
     "Normalizar texto; señalar variantes por similitud SIN fusionarlas automáticamente.",
     "Personas distintas con nombres parecidos podrían fusionarse por error."],
    ["DIRECTOR",
     "La variable con más faltantes; mismo riesgo que SUPERVISOR.",
     "Igual que SUPERVISOR; documentar que el faltante puede ser una vacante real.",
     "Asumir que 'falta' equivale a 'error de captura' sería incorrecto."],
    ["NIVEL",
     "Constante tras el filtro (DIVERSIFICADO).",
     "Conservar como categórica; sirve de control del filtro.",
     "Ninguno."],
    ["SECTOR",
     "Dominio limpio.",
     "Normalizar texto y validar categorías observadas.",
     "Ninguno."],
    ["AREA",
     "Categoría 'SIN ESPECIFICAR' fuera del dominio binario URBANA/RURAL.",
     "Recodificar 'SIN ESPECIFICAR' como NA.",
     "Podría significar 'no aplica' y no un dato faltante; se documenta la decisión."],
    ["STATUS",
     "Categorías poco frecuentes pero administrativamente válidas.",
     "Normalizar texto; NO fusionar categorías raras.",
     "Agruparlas perdería información administrativa real."],
    ["MODALIDAD",
     "Dominio limpio.",
     "Normalizar texto y validar categorías.",
     "Ninguno."],
    ["JORNADA",
     "Categorías raras sin verificar si son reales.",
     "Cruzar con PLAN antes de decidir cualquier recodificación.",
     "Podría borrarse información válida de planes a distancia / fin de semana."],
    ["PLAN",
     "Categorías poco frecuentes.",
     "Documentar sin fusionar.",
     "Fusionar perdería granularidad relevante para el análisis."],
    ["DEPARTAMENTAL",
     "Subdivisión administrativa (Guatemala en 4 zonas, Quiché en 2).",
     "Validar que siempre corresponda al DEPARTAMENTO del registro.",
     "Ninguno; es una verificación, no una modificación."],
], columns=["Variable", "a. Problema encontrado", "b. Regla de corrección", "c. Riesgos"])

plan_limpieza

,Variable,a. Problema encontrado,b. Regla de corrección,c. Riesgos
0,CODIGO,Faltantes provenientes de filas 100% vacías. F...,Eliminar las filas vacías; conservar el código...,Convertirlo a numérico perdería los guiones y ...
1,DISTRITO,Faltantes; dos formatos coexistentes (NN-NNN y...,Registrar el formato en una variable derivada ...,No hay regla segura para convertir un formato ...
2,DEPARTAMENTO,Solo faltantes de filas vacías; dominio limpio.,Validar contra el catálogo oficial de 22 depar...,Ninguno relevante.
3,MUNICIPIO,No cubre los 340 municipios oficiales; posible...,Normalizar a mayúsculas y buscar variantes por...,Hay municipios homónimos en departamentos dist...
4,ESTABLECIMIENTO,Comillas simples y dobles mezcladas; comillas ...,Unificar a comillas dobles y eliminar comillas...,Cambio cosmético; se documenta que no altera e...
5,DIRECCION,"Minúsculas mezcladas, espacios múltiples, abre...",Normalizar espacios y pasar a mayúsculas.,El texto libre puede contener nombres propios ...
6,TELEFONO,Alta tasa de faltantes; varios números por cel...,Extraer los números de 8 dígitos: el primero q...,Riesgo más alto del proyecto: se puede descart...
7,SUPERVISOR,Faltantes; posibles variantes de escritura del...,Normalizar texto; señalar variantes por simili...,Personas distintas con nombres parecidos podrí...
8,DIRECTOR,La variable con más faltantes; mismo riesgo qu...,Igual que SUPERVISOR; documentar que el faltan...,Asumir que 'falta' equivale a 'error de captur...
9,NIVEL,Constante tras el filtro (DIVERSIFICADO).,Conservar como categórica; sirve de control de...,Ninguno.


---
### ***ACTIVIDAD 5 — Limpieza del conjunto de datos***

***5.0 Preparación: renombrado de columnas a nombres descriptivos (trabajo conjunto, actividad 9c)***

Antes de aplicar los incisos a–i se renombran las columnas a `snake_case` descriptivo. A partir de
aquí se trabaja sobre `df`; `df_raw` se conserva intacto para el informe comparativo de la actividad 8.

In [15]:
COLUMN_RENAME = {
    "CODIGO": "codigo_establecimiento",
    "DISTRITO": "codigo_distrito",
    "DEPARTAMENTO": "departamento",
    "MUNICIPIO": "municipio",
    "ESTABLECIMIENTO": "nombre_establecimiento",
    "DIRECCION": "direccion",
    "TELEFONO": "telefono",
    "SUPERVISOR": "supervisor",
    "DIRECTOR": "director",
    "NIVEL": "nivel",
    "SECTOR": "sector",
    "AREA": "area",
    "STATUS": "estado",
    "MODALIDAD": "modalidad",
    "JORNADA": "jornada",
    "PLAN": "plan_educativo",
    "DEPARTAMENTAL": "direccion_departamental",
}

df = df_raw.rename(columns=COLUMN_RENAME).copy()
COLUMNAS_ORIGINALES = list(COLUMN_RENAME.values())

print("Columnas del conjunto de trabajo:")
print(list(df.columns))

Columnas del conjunto de trabajo:
['codigo_establecimiento', 'codigo_distrito', 'departamento', 'municipio', 'nombre_establecimiento', 'direccion', 'telefono', 'supervisor', 'director', 'nivel', 'sector', 'area', 'estado', 'modalidad', 'jornada', 'plan_educativo', 'direccion_departamental']


---
***5a — Valores faltantes, NA, cadenas vacías, espacios en blanco y placeholders***

Tratamiento aplicado:

1. **Filas 100 % vacías** → se eliminan (no aportan información y distorsionan los conteos de
   duplicados y de faltantes).
2. **Cadenas vacías (`""`) y celdas con solo espacios** → se convierten a `NA`.
3. **Placeholders de nulo** (`"N/A"`, `"NA"`, `"NULL"`, `"-"`, `"."`, `"SIN DATO"`, `"S/D"`,
   `"NINGUNO"`, `"XXX"`, `"0"`) → se convierten a `NA`, comparando en mayúsculas y sin espacios
   para que también capturen variantes como `" n/a "`.

**Decisión:** los faltantes **no se imputan**. Un establecimiento sin director registrado puede
tener una vacante real; inventar un valor introduciría un error que el analista no podría detectar.
Se conservan como `NA` explícito y se documentan en el libro de códigos.

In [16]:
# 1. Eliminar filas completamente vacías
df = df.dropna(how="all").copy()

# 2. Convertir cadenas vacías y espacios en blanco a NA
columnas_texto = df.select_dtypes(include="object").columns

df[columnas_texto] = (
    df[columnas_texto]
    .replace(r"^\s*$", pd.NA, regex=True)
)

# 3. Placeholders que representan valores nulos
PLACEHOLDERS_NA = {
    "N/A",
    "NA",
    "NULL",
    "-",
    "--",
    "SIN DATO",
    "S/D",
    "NINGUNO",
    "XXX",
    "@"
}

# Comparar ignorando mayúsculas y espacios
for columna in columnas_texto:

    mascara = (
        df[columna]
        .astype("string")
        .str.strip()
        .str.upper()
        .isin(PLACEHOLDERS_NA)
    )

    df.loc[mascara, columna] = pd.NA

C:\Users\josep\AppData\Local\Temp\ipykernel_77720\3818000154.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include="object").columns


In [17]:
# Estado de los faltantes tras el tratamiento de 5a
print("Filas:", len(df))
df.isna().sum().sort_values(ascending=False)

Filas: 9706


director                   1247
telefono                    682
supervisor                  304
codigo_distrito             301
direccion                    70
nombre_establecimiento        3
codigo_establecimiento        0
municipio                     0
departamento                  0
nivel                         0
sector                        0
area                          0
estado                        0
modalidad                     0
jornada                       0
plan_educativo                0
direccion_departamental       0
dtype: int64

Tal como se puede observar en la suma de valores nulos por variable, las columnas que contenían únicamente registros totalmente vacíos desaparecieron, mientras las otras anomalías se normalizaron al convertirlos a NA.


In [18]:
for c in columnas_texto:

    vacias = (
        df[c]
        .astype("string")
        .str.fullmatch(r"\s*", na=False)
        .sum()
    )

    if vacias > 0:
        print(c, vacias)

---
***5b — Tipo de dato apropiado de cada variable***


Como los datos vienen de una tabla HTML, **todo llegó como `object`**. Se asignan tipos explícitos:

- **`string`** para identificadores y texto libre (`codigo_establecimiento`, `codigo_distrito`,
  `nombre_establecimiento`, `direccion`, `telefono`, `supervisor`, `director`).
  `telefono` y los códigos se mantienen como texto **a propósito**: no se opera aritméticamente con
  ellos y convertirlos a entero eliminaría guiones y ceros a la izquierda.
- **`category`** para las variables con dominio cerrado (`departamento`, `municipio`, `nivel`,
  `sector`, `area`, `estado`, `modalidad`, `jornada`, `plan_educativo`,
  `direccion_departamental`). Se aplica **al final** del pipeline, porque las categorías todavía
  cambian en 5c–5f.
- **`bool`** para las variables derivadas que se crean en 5i.

In [19]:
# ============================================
# 5b. Tipo de dato apropiado de cada variable
# ============================================

# Variables de texto libre e identificadores
VARIABLES_STRING = [
    "codigo_establecimiento",
    "codigo_distrito",
    "nombre_establecimiento",
    "direccion",
    "telefono",
    "supervisor",
    "director",
]

# Variables categóricas (se convertirán a category al finalizar la limpieza)
VARIABLES_CATEGORICAS = [
    "departamento",
    "municipio",
    "nivel",
    "sector",
    "area",
    "estado",
    "modalidad",
    "jornada",
    "plan_educativo",
    "direccion_departamental",
]

# Convertir texto e identificadores a string
df[VARIABLES_STRING] = df[VARIABLES_STRING].astype("string")

# Mantener las variables categóricas como string durante la limpieza
df[VARIABLES_CATEGORICAS] = df[VARIABLES_CATEGORICAS].astype("string")

Importante recordar que las variables categóricas se convertirán de string a category al finalizr la limpieza.

---
***5c — Normalización de texto***


Se aplica a **todas** las columnas de texto, en este orden:

| Paso | Qué corrige |
|---|---|
| `unicodedata.normalize("NFKC", ...)` | caracteres compuestos y variantes de ancho completo |
| eliminación de categorías `Cc`/`Cf` | caracteres invisibles y de control (saltos de línea, BOM, zero-width space) |
| `re.sub(r"\s+", " ", ...)` | espacios múltiples internos, tabuladores |
| `.strip()` | espacios al inicio y al final |
| unificación de comillas tipográficas (`“ ” ‘ ’`) a `"` | signos de puntuación inconsistentes |
| `.upper()` en las columnas donde aplica | uso consistente de mayúsculas |

**Sobre las tildes:** se **conservan** en el texto visible (el nombre correcto es *Sololá*, no
*Solola*). Para comparar valores se usa una función auxiliar `quitar_tildes()` que normaliza solo
la copia usada en la comparación. Así se evita corromper los nombres propios y a la vez se detectan
categorías que difieren únicamente por un acento.

In [20]:
import re
import unicodedata
import pandas as pd

# ============================================
# 5c. Normalización general del texto
# ============================================

# Columnas de texto
COLUMNAS_TEXTO = df.select_dtypes(include=["object", "string"]).columns


# --------------------------------------------------
# Función auxiliar para comparaciones (NO modifica df)
# --------------------------------------------------
def quitar_tildes(texto):
    """
    Elimina tildes únicamente para comparaciones.
    No debe utilizarse para modificar el DataFrame.
    """
    if pd.isna(texto):
        return texto

    texto = unicodedata.normalize("NFD", str(texto))

    return "".join(
        c
        for c in texto
        if unicodedata.category(c) != "Mn"
    )


# --------------------------------------------------
# Función de normalización general
# --------------------------------------------------
def normalizar_texto(texto):

    if pd.isna(texto):
        return pd.NA

    texto = str(texto)

    # 1. Unicode NFC/NFKC
    texto = unicodedata.normalize("NFKC", texto)

    # 2. Eliminar caracteres invisibles y de control
    texto = "".join(
        c
        for c in texto
        if unicodedata.category(c) not in {"Cc", "Cf"}
    )

    # 3. Espacios múltiples
    texto = re.sub(r"\s+", " ", texto)

    # 4. Espacios al inicio y final
    texto = texto.strip()

    # 5. Unificar comillas tipográficas
    texto = (
        texto
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    return texto


# Aplicar a todas las columnas de texto
for columna in COLUMNAS_TEXTO:
    df[columna] = df[columna].map(normalizar_texto)

#### Conversión a mayúsculas
La conversión omitirá las variables codigo_establecimiento, codigo_distrito y telefono debido a que son identificadores.



In [21]:
COLUMNAS_MAYUSCULAS = [
    "departamento",
    "municipio",
    "nombre_establecimiento",
    "direccion",
    "supervisor",
    "director",
    "nivel",
    "sector",
    "area",
    "estado",
    "modalidad",
    "jornada",
    "plan_educativo",
    "direccion_departamental"
]

df[COLUMNAS_MAYUSCULAS] = (
    df[COLUMNAS_MAYUSCULAS]
    .apply(lambda s: s.str.upper())
)

In [22]:
df[VARIABLES_STRING] = df[VARIABLES_STRING].astype("string")
df[VARIABLES_CATEGORICAS] = df[VARIABLES_CATEGORICAS].astype("string")

df.dtypes.to_frame("dtype")

,dtype
codigo_establecimiento,string
codigo_distrito,string
departamento,string
municipio,string
nombre_establecimiento,string
direccion,string
telefono,string
supervisor,string
director,string
nivel,string


---
***5d — Consistencia de categorías***


Se busca detectar categorías que **representan el mismo valor escrito de formas distintas**
(`GUATEMALA` / `Guatemala` / `Guatemla`). El método:

1. Para cada variable categórica se calcula la clave `normalizar_para_comparar()` (sin tildes ni
   puntuación) y se agrupan las categorías que colapsan a la misma clave → **variantes seguras**,
   se unifican a la forma más frecuente.
2. Para `municipio` se usa además **similitud de cadenas (RapidFuzz)**, pero **dentro de cada
   departamento**: hay municipios homónimos en departamentos distintos y unificarlos entre
   departamentos sería un error grave.
3. Las categorías poco frecuentes pero **legítimamente distintas** (`TEMPORAL NOMBRAMIENTO`,
   planes a distancia, etc.) **no se fusionan**: se documentan.

In [23]:
# --- Paso 1: variantes que colapsan a la misma clave normalizada ---
import re
import unicodedata
import pandas as pd

VARIABLES_CATEGORICAS = [
    "departamento",
    "municipio",
    "nivel",
    "sector",
    "area",
    "estado",
    "modalidad",
    "jornada",
    "plan_educativo",
    "direccion_departamental",
]

def normalizar_para_comparar(texto):

    if pd.isna(texto):
        return pd.NA

    texto = str(texto)

    texto = unicodedata.normalize("NFKD", texto)

    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

    texto = texto.upper()

    texto = re.sub(r"[^\w\s]", "", texto)

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


for variable in VARIABLES_CATEGORICAS:

    serie = df[variable].dropna()

    claves = serie.map(normalizar_para_comparar)

    tabla = (
        pd.DataFrame({
            "valor": serie,
            "clave": claves
        })
        .groupby("clave")["valor"]
        .agg(lambda x: x.value_counts().index[0])
    )

    mapa = tabla.to_dict()

    df[variable] = (
        df[variable]
        .map(lambda x: mapa.get(normalizar_para_comparar(x), x)
             if pd.notna(x) else x)
    )

El siguiente paso no modifica al dataframe. Solo genera una tabla para revisión.

In [24]:
# --- Paso 2: variantes de MUNICIPIO por similitud de cadenas, dentro de cada departamento ---
from rapidfuzz import fuzz

municipios_similares = []

for departamento, grupo in df.groupby("departamento"):

    municipios = sorted(
        grupo["municipio"]
        .dropna()
        .unique()
    )

    for i in range(len(municipios)):

        for j in range(i+1, len(municipios)):

            score = fuzz.ratio(
                municipios[i],
                municipios[j]
            )

            if score >= 90:

                municipios_similares.append({
                    "departamento": departamento,
                    "municipio_1": municipios[i],
                    "municipio_2": municipios[j],
                    "similitud": score
                })

revision_municipios = pd.DataFrame(municipios_similares)

if revision_municipios.empty:
    print("No se encontraron municipios con similitud ≥ 90 %. No hay cambios para revisar.")
else:
    revision_municipios = revision_municipios.sort_values(
        ["departamento", "similitud"],
        ascending=[True, False]
    )

display(revision_municipios)



No se encontraron municipios con similitud ≥ 90 %. No hay cambios para revisar.


""


**Limitación documentada.** La detección de variantes de `municipio` se basa únicamente en
similitud de cadenas dentro de cada departamento y **no explica** la diferencia entre los
municipios presentes y los 340 oficiales del país. Lo más probable es que esa brecha corresponda a
municipios reales **sin ningún establecimiento de nivel Diversificado**, no a errores de captura.
No se transcribió de memoria un catálogo completo de los 340 municipios porque el riesgo de
introducir errores (y marcar como "inválidos" municipios que sí existen) es mayor que el beneficio;
queda como verificación pendiente contra la fuente oficial del INE. `departamento`, en cambio,
sí se valida por completo contra los 22 departamentos oficiales.

In [25]:
# Categorías finales de cada variable categórica (revisión manual del dominio)
for variable in VARIABLES_CATEGORICAS:

    print("\n")
    print("="*70)
    print(variable.upper())
    print("="*70)

    display(
        df[variable]
        .value_counts(dropna=False)
    )



DEPARTAMENTO


departamento
GUATEMALA         1908
SAN MARCOS         724
ESCUINTLA          708
HUEHUETENANGO      591
QUETZALTENANGO     551
PETEN              516
ALTA VERAPAZ       475
SUCHITEPEQUEZ      437
CHIMALTENANGO      435
SACATEPEQUEZ       430
IZABAL             413
JUTIAPA            392
RETALHULEU         364
QUICHE             322
CHIQUIMULA         239
SANTA ROSA         213
SOLOLA             192
JALAPA             183
BAJA VERAPAZ       171
EL PROGRESO        158
ZACAPA             156
TOTONICAPAN        128
Name: count, dtype: int64



MUNICIPIO


municipio
MIXCO                562
VILLA NUEVA          446
QUETZALTENANGO       281
RETALHULEU           204
MAZATENANGO          200
                    ... 
SAN JUAN ATITAN        1
SAN GASPAR IXCHIL      1
PETATAN                1
CHICHE                 1
PATZITE                1
Name: count, Length: 330, dtype: int64



NIVEL


...

jornada
DOBLE          3217
VESPERTINA     3092
MATUTINA       2084
SIN JORNADA     952
NOCTURNA        296
INTERMEDIA       65
Name: count, dtype: int64



PLAN_EDUCATIVO


plan_educativo
DIARIO(REGULAR)                          6038
FIN DE SEMANA                            2399
SEMIPRESENCIAL (FIN DE SEMANA)            428
SEMIPRESENCIAL (UN DÍA A LA SEMANA)       426
A DISTANCIA                               148
SEMIPRESENCIAL                            100
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      58
VIRTUAL A DISTANCIA                        53
SABATINO                                   30
DOMINICAL                                  18
MIXTO                                       3
IRREGULAR                                   3
INTERCALADO                                 2
Name: count, dtype: int64



DIRECCION_DEPARTAMENTAL


direccion_departamental
GUATEMALA SUR          752
GUATEMALA OCCIDENTE    724
SAN MARCOS             724
ESCUINTLA              708
HUEHUETENANGO          591
QUETZALTENANGO         551
PETÉN                  516
ALTA VERAPAZ           475
SUCHITEPÉQUEZ          437
CHIMALTENANGO          435
SACATEPÉQUEZ           430
IZABAL                 413
JUTIAPA                392
RETALHULEU             364
GUATEMALA ORIENTE      262
QUICHÉ                 261
CHIQUIMULA             239
SANTA ROSA             213
SOLOLÁ                 192
JALAPA                 183
BAJA VERAPAZ           171
GUATEMALA NORTE        170
EL PROGRESO            158
ZACAPA                 156
TOTONICAPÁN            128
QUICHÉ NORTE            61
Name: count, dtype: int64

Según el conteo de categorías por variable, no se encontraron anomalías en cuanto al registro de datos. La normalización de los pasos anteriores pudo haber coadyuvado a esto. Ahora, se inspeccionaron principalmente los departamentos y municipios. De ellos solamente se encontró un posible error que resultó ser ignorancia de parte nuestra porque había un conteo de centro educativo en San Juan Atitán. Se pensaba que era Atitlán, pero estaba escrito correctamente.

---
***5e — Formatos (teléfonos, códigos, direcciones, nombres)***

### Teléfonos
En Guatemala el número fijo/móvil tiene **8 dígitos**. El campo original mezcla números, guiones,
comas y texto, y a veces trae **más de un teléfono en la misma celda**. La regla:

1. Extraer todas las secuencias de dígitos de la celda.
2. Quedarse con las de exactamente 8 dígitos: la primera va a `telefono`, la segunda a `telefono_2`.
3. Si ninguna califica, `telefono` queda en `NA` (no se inventa ni se rellena con ceros).

### Códigos
- `codigo_establecimiento`: se valida el formato `NN-NN-NNNN-NN` (es la llave primaria).
- `codigo_distrito`: coexisten `NN-NNN` y `NN-NN-NNNN`. **No hay forma segura de convertir uno en
  otro sin inventar dígitos**, así que se documenta el formato de cada registro en una variable
  derivada y los códigos truncados pasan a `NA`.

### Direcciones y nombres
- `direccion`: espacios y mayúsculas ya se normalizaron en 5c; aquí se corrige el único patrón de
  formato que quedaba: espacios antes de una coma o un punto (`"ZONA 12 ,"` → `"ZONA 12,"`).
- `nombre_establecimiento`: se corrigen las comillas dobles (`"`) que quedaron **sin cerrar**
  (número impar de `"` en el registro). Como no hay forma de saber con certeza dónde faltaba la
  comilla, se eliminan todas las comillas dobles de ese registro puntual en lugar de inventar una
  posición. **Las comillas simples (`'`) no se tocan**: en este conjunto son ortografía real de
  nombres en idiomas mayas (`RUK'U'X`, `NALEB'`, `KAQCHIKEL AMAQ'`) o posesivos en inglés
  (`BROWN'S`), no comillas de estilo — unificarlas a comillas dobles corrompería esos nombres.


In [26]:
# ------------------ Teléfonos ------------------
telefono_original = df["telefono"]


def extraer_telefonos(celda):
    """Devuelve (primer_telefono, segundo_telefono, tiene_mas_de_una_secuencia)."""
    if pd.isna(celda):
        return pd.NA, pd.NA, False

    secuencias = re.findall(r"\d+", celda)
    validos = [s for s in secuencias if len(s) == 8]

    primero = validos[0] if len(validos) >= 1 else pd.NA
    segundo = validos[1] if len(validos) >= 2 else pd.NA

    return primero, segundo, len(secuencias) > 1


extraidos = telefono_original.map(extraer_telefonos)

df["telefono"] = pd.Series([t[0] for t in extraidos], index=df.index, dtype="string")
df["telefono_2"] = pd.Series([t[1] for t in extraidos], index=df.index, dtype="string")
df["telefono_multiple"] = pd.Series([t[2] for t in extraidos], index=df.index)
df["telefono_valido"] = df["telefono"].notna()

telefonos_multiples = int(df["telefono_2"].notna().sum())
telefonos_descartados = int((telefono_original.notna() & df["telefono"].isna()).sum())

pd.DataFrame({
    "Métrica": [
        "Celdas con más de una secuencia de dígitos",
        "Segundo teléfono desagregado (telefono_2)",
        "Teléfonos válidos (8 dígitos)",
        "Teléfonos descartados (ninguna secuencia de 8 dígitos)",
    ],
    "Cantidad": [
        int(df["telefono_multiple"].sum()),
        telefonos_multiples,
        int(df["telefono_valido"].sum()),
        telefonos_descartados,
    ],
})


,Métrica,Cantidad
0,Celdas con más de una secuencia de dígitos,123
1,Segundo teléfono desagregado (telefono_2),93
2,Teléfonos válidos (8 dígitos),8961
3,Teléfonos descartados (ninguna secuencia de 8 ...,63


In [27]:
# ------------------ Códigos ------------------
PATRON_CODIGO_ESTABLECIMIENTO = r"\d{2}-\d{2}-\d{4}-\d{2}"
PATRON_DISTRITO_CORTO = r"\d{2}-\d{3}"
PATRON_DISTRITO_LARGO = r"\d{2}-\d{2}-\d{4}"

df["codigo_establecimiento_valido"] = (
    df["codigo_establecimiento"].str.fullmatch(PATRON_CODIGO_ESTABLECIMIENTO).fillna(False)
)

es_formato_corto = df["codigo_distrito"].str.fullmatch(PATRON_DISTRITO_CORTO).fillna(False)
es_formato_largo = df["codigo_distrito"].str.fullmatch(PATRON_DISTRITO_LARGO).fillna(False)

df["codigo_distrito_formato"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[es_formato_corto, "codigo_distrito_formato"] = "NN-NNN"
df.loc[es_formato_largo, "codigo_distrito_formato"] = "NN-NN-NNNN"

# Códigos truncados: tenían un valor, pero no calzan con ninguno de los dos formatos conocidos.
codigo_distrito_truncados = int(
    (df["codigo_distrito"].notna() & df["codigo_distrito_formato"].isna()).sum()
)
df.loc[df["codigo_distrito_formato"].isna(), "codigo_distrito"] = pd.NA

pd.DataFrame({
    "Métrica": [
        "codigo_establecimiento con formato NN-NN-NNNN-NN",
        "codigo_distrito con formato NN-NNN",
        "codigo_distrito con formato NN-NN-NNNN",
        "codigo_distrito truncado (pasado a NA)",
    ],
    "Cantidad": [
        int(df["codigo_establecimiento_valido"].sum()),
        int(es_formato_corto.sum()),
        int(es_formato_largo.sum()),
        codigo_distrito_truncados,
    ],
})


,Métrica,Cantidad
0,codigo_establecimiento con formato NN-NN-NNNN-NN,9706
1,codigo_distrito con formato NN-NNN,3928
2,codigo_distrito con formato NN-NN-NNNN,5461
3,codigo_distrito truncado (pasado a NA),16


In [28]:
# ------------------ Direcciones: espacio antes de puntuación ------------------
direccion_original = df["direccion"]

df["direccion"] = df["direccion"].str.replace(r"\s+([.,])", r"\1", regex=True)

direcciones_ajustadas = int((df["direccion"] != direccion_original).fillna(False).sum())

# ------------------ Nombres: comillas dobles sin cerrar ------------------
# Las comillas simples (') NO se tocan: en este conjunto son ortografía real de nombres en
# idiomas mayas ("RUK'U'X", "NALEB'") o posesivos en inglés ("BROWN'S"), no comillas de estilo.
comillas_desbalanceadas = (df["nombre_establecimiento"].str.count('"') % 2 == 1).fillna(False)

nombres_comillas_corregidas = int(comillas_desbalanceadas.sum())

df.loc[comillas_desbalanceadas, "nombre_establecimiento"] = (
    df.loc[comillas_desbalanceadas, "nombre_establecimiento"].str.replace('"', "", regex=False)
)

pd.DataFrame({
    "Métrica": [
        "Direcciones con espacio antes de puntuación corregido",
        "Nombres con comillas dobles desbalanceadas corregidas",
    ],
    "Cantidad": [direcciones_ajustadas, nombres_comillas_corregidas],
})


,Métrica,Cantidad
0,Direcciones con espacio antes de puntuación co...,15
1,Nombres con comillas dobles desbalanceadas cor...,14


---
***5f — Valores inválidos / fuera de dominio***


| Variable | Dominio esperado | Acción |
|---|---|---|
| `departamento` | los 22 departamentos oficiales | validar; ninguno fuera de catálogo |
| `area` | `{URBANA, RURAL}` | `SIN ESPECIFICAR` → `NA` |
| `telefono` | 8 dígitos | los que no califican ya quedaron en `NA` (5e) |
| `codigo_establecimiento` | `NN-NN-NNNN-NN` | validado en 5e |
| `nivel` | `DIVERSIFICADO` (constante tras el filtro) | validar que no haya otros niveles |

`area = "SIN ESPECIFICAR"` se trata como faltante porque **no pertenece al dominio binario**
y no aporta información. Se documenta la decisión: si el MINEDUC lo usara con el sentido de
"no aplica", el registro seguiría siendo recuperable porque **no se elimina la fila**.

In [29]:
# ------------------ area: valor fuera del dominio {URBANA, RURAL} ------------------
area_sin_especificar = int((df["area"] == "SIN ESPECIFICAR").sum())
df.loc[df["area"] == "SIN ESPECIFICAR", "area"] = pd.NA

# ------------------ departamento: validación contra el catálogo oficial ------------------
departamentos_fuera_de_catalogo = sorted(
    df.loc[df["departamento"].notna() & ~df["departamento"].isin(DEPARTAMENTOS_OFICIALES),
           "departamento"].unique()
)

# ------------------ nivel: debe ser constante DIVERSIFICADO ------------------
niveles_inesperados = sorted(set(df["nivel"].dropna().unique()) - {"DIVERSIFICADO"})

resumen_valores_invalidos = pd.DataFrame({
    "Variable": ["area", "departamento", "nivel", "telefono", "codigo_establecimiento"],
    "Dominio esperado": [
        "{URBANA, RURAL}",
        "22 departamentos oficiales",
        "constante = DIVERSIFICADO",
        "8 dígitos (extracción hecha en 5e)",
        "NN-NN-NNNN-NN (validado en 5e)",
    ],
    "Valores fuera de dominio": [
        area_sin_especificar,
        len(departamentos_fuera_de_catalogo),
        len(niveles_inesperados),
        telefonos_descartados,
        int((~df["codigo_establecimiento_valido"]).sum()),
    ],
    "Tratamiento": [
        "Recodificado a NA",
        "Ninguno pendiente" if not departamentos_fuera_de_catalogo
            else f"Revisar: {departamentos_fuera_de_catalogo}",
        "Ninguno pendiente" if not niveles_inesperados else f"Revisar: {niveles_inesperados}",
        "Ya pasado a NA en 5e",
        "N/A: no se detectaron códigos fuera de formato",
    ],
})
resumen_valores_invalidos


,Variable,Dominio esperado,Valores fuera de dominio,Tratamiento
0,area,"{URBANA, RURAL}",2,Recodificado a NA
1,departamento,22 departamentos oficiales,0,Ninguno pendiente
2,nivel,constante = DIVERSIFICADO,0,Ninguno pendiente
3,telefono,8 dígitos (extracción hecha en 5e),63,Ya pasado a NA en 5e
4,codigo_establecimiento,NN-NN-NNNN-NN (validado en 5e),0,N/A: no se detectaron códigos fuera de formato


---
***5g — Registros duplicados: exactos y parciales***

### Duplicados exactos
Filas idénticas en **todas** sus columnas. Al no aportar información adicional, se eliminan con
`drop_duplicates()` dejando la primera ocurrencia. También se revisa la llave primaria
`codigo_establecimiento`: dos filas con el mismo código son sospechosas aunque el resto de columnas
difieran.

### Duplicados parciales
Se usa **similitud de cadenas (RapidFuzz, `token_sort_ratio`)** sobre `nombre_establecimiento` y
`direccion`, exigiendo que **ambas** superen su umbral (`nombre ≥ 90`, `dirección ≥ 85`; la
dirección usa un umbral algo menor porque suele variar más por números de casa/lote sin dejar de
ser la misma sede), y la comparación se hace **dentro de cada par (departamento, municipio)**.
Justificación de cada decisión de diseño:

- `token_sort_ratio` en lugar de `WRatio`: es insensible al orden de las palabras
  (`"COLEGIO SAN JOSE"` vs `"SAN JOSE, COLEGIO"`) pero mucho menos propenso a los falsos positivos
  que produce `WRatio` con coincidencias parciales de subcadenas.
- Exigir nombre **y** dirección: por sí solo el nombre genera falsos positivos masivos (hay cientos
  de "INSTITUTO NACIONAL DE EDUCACION BASICA" en el país).
- Restringir a (departamento, municipio): dos establecimientos con el mismo nombre en municipios
  distintos son establecimientos distintos, y además reduce el costo del cálculo de O(n²) global a
  la suma de los O(nᵢ²) por municipio.

### No se elimina automáticamente ningún duplicado parcial
Cada grupo se etiqueta cruzándolo con `jornada`, `plan_educativo` y `sector`:

- **`misma_sede_oferta_distinta`** → el grupo tiene más de una combinación de jornada/plan/sector:
  es el **mismo establecimiento** ofreciendo Diversificado en más de una modalidad. **No es un
  error de datos**, es la granularidad real de la fuente.
- **`posible_duplicado_real`** → mismo nombre, misma dirección **y** misma combinación de
  jornada/plan/sector: es el subconjunto que el equipo debe revisar caso por caso.

El resultado queda registrado en las variables derivadas `posible_duplicado`,
`grupo_posible_duplicado` y `tipo_posible_duplicado` (ver 5i).


In [30]:
# ---------- Duplicados exactos ----------
duplicados_exactos = int(df.duplicated(subset=COLUMNAS_ORIGINALES).sum())
df = df.drop_duplicates(subset=COLUMNAS_ORIGINALES).reset_index(drop=True)

# codigo_establecimiento repetido entre registros que sí quedaron distintos: la llave primaria
# no debería repetirse; si aparece, se deja para inspección manual (no se corrige de oficio).
codigos_repetidos = df.loc[df["codigo_establecimiento"].notna(), "codigo_establecimiento"].value_counts()
codigos_repetidos = codigos_repetidos[codigos_repetidos > 1]

print(f"Duplicados exactos eliminados: {duplicados_exactos}")
print(f"Filas restantes: {len(df)}")

if len(codigos_repetidos):
    print(f"\ncodigo_establecimiento repetido en {len(codigos_repetidos)} código(s) (revisión manual):")
    display(df[df["codigo_establecimiento"].isin(codigos_repetidos.index)]
            .sort_values("codigo_establecimiento"))
else:
    print("Ningún codigo_establecimiento se repite entre registros distintos.")


Duplicados exactos eliminados: 0
Filas restantes: 9706
Ningún codigo_establecimiento se repite entre registros distintos.


In [31]:
# ---------- Duplicados parciales (similitud de cadenas) ----------
UMBRAL_NOMBRE = 90
UMBRAL_DIRECCION = 85


def pares_similares_en_grupo(grupo):
    """Compara todo par de registros del grupo por similitud de nombre y dirección."""
    comparable = grupo.dropna(subset=["nombre_establecimiento", "direccion"])
    if len(comparable) < 2:
        return []

    nombres = comparable["nombre_establecimiento"].tolist()
    direcciones = comparable["direccion"].tolist()
    indices = comparable.index.to_numpy()

    similitud_nombre = process.cdist(nombres, nombres, scorer=fuzz.token_sort_ratio)
    similitud_direccion = process.cdist(direcciones, direcciones, scorer=fuzz.token_sort_ratio)

    i, j = np.triu_indices(len(indices), k=1)
    coincide = (similitud_nombre[i, j] >= UMBRAL_NOMBRE) & (similitud_direccion[i, j] >= UMBRAL_DIRECCION)

    return list(zip(indices[i[coincide]], indices[j[coincide]]))


pares_similares = []
for _, grupo in df.groupby(["departamento", "municipio"], observed=True):
    pares_similares.extend(pares_similares_en_grupo(grupo))

print(f"Pares de registros similares dentro del mismo departamento/municipio "
      f"(nombre >= {UMBRAL_NOMBRE}, dirección >= {UMBRAL_DIRECCION}): {len(pares_similares)}")

# Unión de pares transitivos en grupos (Union-Find): si A~B y B~C, los tres quedan en un grupo.
padre_grupo = {i: i for i in df.index}


def raiz(i):
    while padre_grupo[i] != i:
        padre_grupo[i] = padre_grupo[padre_grupo[i]]
        i = padre_grupo[i]
    return i


for a, b in pares_similares:
    ra, rb = raiz(a), raiz(b)
    if ra != rb:
        padre_grupo[ra] = rb

miembros_por_raiz = {}
for i in df.index:
    miembros_por_raiz.setdefault(raiz(i), []).append(i)

grupos_similares = {r: filas for r, filas in miembros_por_raiz.items() if len(filas) > 1}

print(f"Grupos de posibles duplicados formados: {len(grupos_similares)}")


Pares de registros similares dentro del mismo departamento/municipio (nombre >= 90, dirección >= 85): 5948
Grupos de posibles duplicados formados: 1756


In [32]:
# ---------- Clasificación de cada grupo: ¿oferta distinta o duplicado real? ----------
df["posible_duplicado"] = False
df["grupo_posible_duplicado"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df["tipo_posible_duplicado"] = pd.Series(pd.NA, index=df.index, dtype="string")

for numero_grupo, filas in enumerate(grupos_similares.values(), start=1):
    df.loc[filas, "posible_duplicado"] = True
    df.loc[filas, "grupo_posible_duplicado"] = numero_grupo

    ofertas_distintas = df.loc[filas, ["jornada", "plan_educativo", "sector"]].drop_duplicates()
    df.loc[filas, "tipo_posible_duplicado"] = (
        "misma_sede_oferta_distinta" if len(ofertas_distintas) > 1 else "posible_duplicado_real"
    )

df["tipo_posible_duplicado"] = df["tipo_posible_duplicado"].astype("category")

posibles_duplicados_totales = int(df["posible_duplicado"].sum())
posibles_duplicados_reales = int((df["tipo_posible_duplicado"] == "posible_duplicado_real").sum())

pd.DataFrame({
    "Métrica": [
        "Grupos de posibles duplicados",
        "Registros marcados como posible_duplicado",
        "misma_sede_oferta_distinta",
        "posible_duplicado_real",
    ],
    "Cantidad": [
        len(grupos_similares),
        posibles_duplicados_totales,
        int((df["tipo_posible_duplicado"] == "misma_sede_oferta_distinta").sum()),
        posibles_duplicados_reales,
    ],
})


,Métrica,Cantidad
0,Grupos de posibles duplicados,1756
1,Registros marcados como posible_duplicado,5001
2,misma_sede_oferta_distinta,4486
3,posible_duplicado_real,515


In [33]:
# Muestra de grupos clasificados como duplicado real, para la revisión caso por caso
if posibles_duplicados_reales:
    grupos_reales = (
        df.loc[df["tipo_posible_duplicado"] == "posible_duplicado_real", "grupo_posible_duplicado"]
        .unique()
    )
    display(
        df[df["grupo_posible_duplicado"].isin(grupos_reales)]
        .sort_values("grupo_posible_duplicado")
        [["grupo_posible_duplicado", "codigo_establecimiento", "nombre_establecimiento",
          "direccion", "jornada", "plan_educativo", "sector"]]
    )
else:
    print("No se encontraron grupos clasificados como posible_duplicado_real.")


,grupo_posible_duplicado,codigo_establecimiento,nombre_establecimiento,direccion,jornada,plan_educativo,sector
88,12,01-03-0132-46,"CENTRO EDUCATIVO "" EL PROGRESO """,5A. CALLE 2-50 ZONA 2 GUATEMALA,DOBLE,FIN DE SEMANA,PRIVADO
89,12,01-03-0133-46,"CENTRO EDUCATIVO "" EL PROGRESO""",5A. CALLE 2-50 ZONA 2 GUATEMALA,DOBLE,FIN DE SEMANA,PRIVADO
86,12,01-03-0129-46,"CENTRO EDUCATIVO "" EL PROGRESO""",5A. CALLE 2-50 ZONA 2 GUATEMALA,DOBLE,FIN DE SEMANA,PRIVADO
155,25,01-05-0071-46,LICEO SAN FRANCISCO DE ASÍS,"2DA. CALLE 2-45, PUEBLO NUEVO NORTE",MATUTINA,DIARIO(REGULAR),PRIVADO
178,25,01-05-6543-46,LICEO SAN FRANCISCO DE ASIS,"PUEBLO NUEVO NORTE, 2DA. CALLE 2-45",MATUTINA,DIARIO(REGULAR),PRIVADO
...,...,...,...,...,...,...,...
9521,1720,22-06-0020-46,LICEO YUPANO,BARRIO EL CENTRO,DOBLE,FIN DE SEMANA,PRIVADO
9579,1731,22-11-0079-46,INSTITUTO PARTICULAR MIXTO DE FORMACION Y DESA...,ENTRADA AL MUNICIPIO DE COMAPA,DOBLE,FIN DE SEMANA,PRIVADO
9578,1731,22-11-0078-46,INSTITUTO PARTICULAR MIXTO DE FORMACIÓN Y DESA...,ENTRADA AL MUNICIPIO DE COMAPA,DOBLE,FIN DE SEMANA,PRIVADO
9700,1756,22-17-0039-46,"COLEGIO INTEGRAL DE COMPUTACIÓN ""EDUCARE""",7A. CALLE 4-32 ZONA 1,DOBLE,FIN DE SEMANA,PRIVADO


### Decisión documentada sobre los duplicados

- **Duplicados exactos:** se eliminan. Filas idénticas en todas sus columnas no aportan información
  y sesgarían cualquier conteo por municipio o por sector.
- **Duplicados parciales:** **ninguno se elimina automáticamente**, tal como exige la guía. La
  mayoría de los registros marcados corresponden a `misma_sede_oferta_distinta` — el mismo
  establecimiento con más de una jornada o plan — y eliminarlos destruiría información real sobre
  la oferta educativa. Solo el subconjunto `posible_duplicado_real` queda como lista priorizada de
  revisión manual, y se entrega marcado en el conjunto limpio para que el analista decida.
- **`codigo_establecimiento` repetido:** si aparecieran códigos repetidos, se listan arriba para
  inspección; al ser la llave primaria de la fuente, un código repetido con datos distintos indica
  un problema en el origen que no debe resolverse automáticamente.

---
***5h — Consistencia entre variables***


Se verifica que los valores de distintas variables **no se contradigan entre sí**:

1. **`departamento` ↔ `direccion_departamental`**: la dirección departamental es la subdivisión
   administrativa del MINEDUC; siempre debe empezar con el nombre del departamento
   (Guatemala se divide en Norte / Sur / Oriente / Occidente y Quiché en Quiché / Quiché Norte).
2. **`departamento` ↔ `codigo_establecimiento`**: los dos primeros dígitos del código son el
   código del departamento; deben ser constantes dentro de cada departamento.
3. **`codigo_establecimiento` ↔ `codigo_distrito`**: ambos deben empezar con el mismo código de
   departamento.
4. **`departamento` ↔ `municipio`**: cada municipio debe pertenecer a un solo departamento
   (salvo homónimos legítimos, que se listan explícitamente).
5. **`jornada` ↔ `plan_educativo`**: se cruza para confirmar si las jornadas poco frecuentes
   corresponden a planes válidos (fin de semana, a distancia) antes de considerarlas un error.
6. **`area` ↔ `municipio`**: un municipio con establecimientos marcados como urbanos y rurales es
   normal; se reporta solo como contexto.

In [34]:
inconsistencias = []

# --- 1. departamento vs direccion_departamental ---
sub = df[["departamento", "direccion_departamental"]].dropna()
dep_n = sub["departamento"].map(quitar_tildes)
dpal_n = sub["direccion_departamental"].map(quitar_tildes)
mask_1 = ~pd.Series([d.startswith(p) for d, p in zip(dpal_n, dep_n)], index=sub.index)
inconsistencias.append(("departamento vs direccion_departamental", int(mask_1.sum()), len(sub)))

# --- 2. departamento vs prefijo del codigo_establecimiento ---
prefijo = df["codigo_establecimiento"].str.slice(0, 2)
# El prefijo "esperado" de cada departamento es el más frecuente dentro de ese
# departamento (no se transcribe de memoria: se deduce de los propios datos).
mapa_prefijo = {}
for dep, g in df.dropna(subset=["departamento"]).groupby("departamento", observed=True):
    modas = prefijo.loc[g.index].mode(dropna=True)
    if not modas.empty:
        mapa_prefijo[dep] = modas.iloc[0]
prefijo_esperado = df["departamento"].map(mapa_prefijo)
mask_2 = (prefijo.notna() & prefijo_esperado.notna() & (prefijo != prefijo_esperado)).fillna(False)
inconsistencias.append(("departamento vs prefijo de codigo_establecimiento",
                        int(mask_2.sum()), int(prefijo.notna().sum())))

# --- 3. codigo_establecimiento vs codigo_distrito ---
mask_3 = (df["codigo_establecimiento"].notna() & df["codigo_distrito"].notna()
          & (df["codigo_establecimiento"].str.slice(0, 2) != df["codigo_distrito"].str.slice(0, 2))
          ).fillna(False)
inconsistencias.append(("codigo_establecimiento vs codigo_distrito", int(mask_3.sum()),
                        int((df["codigo_establecimiento"].notna() & df["codigo_distrito"].notna()).sum())))

# --- 4. municipios que aparecen en más de un departamento (homónimos) ---
homonimos = (df.dropna(subset=["municipio", "departamento"])
               .groupby("municipio", observed=True)["departamento"].nunique())
homonimos = homonimos[homonimos > 1]
inconsistencias.append(("municipios presentes en más de un departamento (homónimos)",
                        int(len(homonimos)), int(df["municipio"].nunique())))

# --- 5. jornada vs plan_educativo ---
cruce_jornada_plan = pd.crosstab(df["jornada"], df["plan_educativo"], dropna=False)
combinaciones_observadas = int((cruce_jornada_plan > 0).sum().sum())
inconsistencias.append(("combinaciones jornada x plan_educativo observadas",
                        combinaciones_observadas, len(df)))

# --- 6. area vs municipio (contexto, no es un error) ---
mixtos = (df.dropna(subset=["area"])
            .groupby(["departamento", "municipio"], observed=True)["area"].nunique())
inconsistencias.append(("municipios con establecimientos urbanos y rurales a la vez",
                        int((mixtos > 1).sum()), int(len(mixtos))))

resumen_consistencia = pd.DataFrame(
    inconsistencias, columns=["Chequeo", "Casos detectados", "Registros comparados"])
resumen_consistencia

,Chequeo,Casos detectados,Registros comparados
0,departamento vs direccion_departamental,0,9706
1,departamento vs prefijo de codigo_establecimiento,0,9706
2,codigo_establecimiento vs codigo_distrito,28,9389
3,municipios presentes en más de un departamento...,6,330
4,combinaciones jornada x plan_educativo observadas,35,9706
5,municipios con establecimientos urbanos y rura...,272,336


In [35]:
# Detalle de cualquier inconsistencia real detectada en los chequeos 1, 2 y 3
if mask_1.any():
    print("Registros donde direccion_departamental no corresponde al departamento:")
    display(df.loc[mask_1[mask_1].index, ["departamento", "direccion_departamental"]]
            .drop_duplicates())

if mask_2.any():
    print("\nRegistros con prefijo de código que no coincide con su departamento:")
    display(df.loc[mask_2, ["departamento", "codigo_establecimiento"]].drop_duplicates())

if mask_3.any():
    print("\nRegistros donde codigo_establecimiento y codigo_distrito no coinciden:")
    display(df.loc[mask_3, ["departamento", "codigo_establecimiento", "codigo_distrito"]])

if not (mask_1.any() or mask_2.any() or mask_3.any()):
    print("Los chequeos 1, 2 y 3 no detectaron ninguna contradicción.")


Registros donde codigo_establecimiento y codigo_distrito no coinciden:


,departamento,codigo_establecimiento,codigo_distrito
245,GUATEMALA,01-06-3001-46,99-001
1658,GUATEMALA,01-15-9791-46,99-001
2640,CHIMALTENANGO,04-01-2671-46,25-000
3081,ESCUINTLA,05-01-2417-46,25-000
3667,SANTA ROSA,06-01-1515-46,25-000
3896,SOLOLA,07-01-2687-46,25-000
4086,TOTONICAPAN,08-01-2170-46,25-000
4378,QUETZALTENANGO,09-01-3536-46,25-000
4891,SUCHITEPEQUEZ,10-01-2213-46,25-000
5341,RETALHULEU,11-01-1253-46,25-000


In [36]:
# Detalle de los municipios homónimos (no son un error: son nombres repetidos legítimos)
if len(homonimos):
    display(df[df["municipio"].isin(homonimos.index)]
            .groupby(["municipio", "departamento"], observed=True)
            .size().rename("establecimientos").reset_index()
            .sort_values(["municipio", "departamento"]))
else:
    print("No hay municipios con el mismo nombre en departamentos distintos.")

,municipio,departamento,establecimientos
0,LA DEMOCRACIA,ESCUINTLA,25
1,LA DEMOCRACIA,HUEHUETENANGO,35
2,LA LIBERTAD,HUEHUETENANGO,10
3,LA LIBERTAD,PETEN,77
4,SAN JOSE,ESCUINTLA,59
5,SAN JOSE,PETEN,6
6,SAN LORENZO,SAN MARCOS,26
7,SAN LORENZO,SUCHITEPEQUEZ,3
8,SAN PEDRO SACATEPEQUEZ,GUATEMALA,22
9,SAN PEDRO SACATEPEQUEZ,SAN MARCOS,70


In [37]:
# Detalle del cruce jornada x plan_educativo (chequeo 5)
print("Cruce jornada x plan_educativo:")
display(cruce_jornada_plan)

print(f"\nMunicipios con establecimientos urbanos y rurales a la vez: "
      f"{int((mixtos > 1).sum())} de {len(mixtos)} (esperado y normal).")

Cruce jornada x plan_educativo:


plan_educativo,A DISTANCIA,DIARIO(REGULAR),DOMINICAL,FIN DE SEMANA,INTERCALADO,IRREGULAR,MIXTO,SABATINO,SEMIPRESENCIAL,SEMIPRESENCIAL (DOS DÍAS A LA SEMANA),SEMIPRESENCIAL (FIN DE SEMANA),SEMIPRESENCIAL (UN DÍA A LA SEMANA),VIRTUAL A DISTANCIA
jornada,,,,,,,,,,,,,
DOBLE,79,713,13,2283,2,3,0,15,95,0,0,0,14
INTERMEDIA,2,59,1,3,0,0,0,0,0,0,0,0,0
MATUTINA,29,1943,4,73,0,0,1,10,4,0,0,0,20
NOCTURNA,0,294,0,1,0,0,1,0,0,0,0,0,0
SIN JORNADA,38,0,0,2,0,0,0,0,0,58,428,426,0
VESPERTINA,0,3029,0,37,0,0,1,5,1,0,0,0,19



Municipios con establecimientos urbanos y rurales a la vez: 272 de 336 (esperado y normal).


### Inconsistencias encontradas y decisión

Las verificaciones anteriores documentan el estado real del conjunto. **Ninguna inconsistencia se
corrige de forma automática**, por dos razones:

1. Cuando dos variables se contradicen no hay manera de saber cuál de las dos es la incorrecta sin
   consultar la fuente; corregir una a partir de la otra propagaría el error en lugar de eliminarlo.
2. Los "conflictos" de `municipio` entre departamentos son **homónimos legítimos** del sistema
   administrativo guatemalteco (por ejemplo, hay municipios con el mismo nombre en departamentos
   distintos), no errores de captura.

Los chequeos 5 y 6 no buscan errores sino **contexto**: confirman que las jornadas poco frecuentes
corresponden a planes válidos y que la mezcla urbano/rural dentro de un municipio es lo esperado.

Todos los casos detectados quedan registrados en la tabla anterior y en el registro de
transformaciones de la actividad 6.

---
***5i — Variables derivadas***


Se crearon **9 variables derivadas**. Ninguna reemplaza a una variable original: todas son
**metadatos de calidad** que permiten al analista filtrar el conjunto según el nivel de confianza
que necesite, sin tener que repetir el proceso de limpieza. Todas aparecen en el libro de códigos.

| Variable | Cómo se calcula | Para qué sirve |
|---|---|---|
| `telefono_2` | segundo número de 8 dígitos hallado en la celda original | evita perder el segundo contacto cuando la celda traía varios |
| `telefono_valido` | `telefono.notna()` tras la extracción | filtrar rápidamente los registros contactables |
| `telefono_multiple` | la celda original contenía más de una secuencia de dígitos | señala qué registros hubo que desagregar (trazabilidad) |
| `codigo_establecimiento_valido` | cumple `NN-NN-NNNN-NN` | control de integridad de la llave primaria |
| `codigo_distrito_formato` | `"NN-NNN"` / `"NN-NN-NNNN"` / `NA` | documenta cuál de las dos convenciones usa cada registro, sin alterar el valor |
| `posible_duplicado` | pertenece a un grupo de similitud nombre+dirección | permite excluir candidatos a duplicado en un análisis conservador |
| `grupo_posible_duplicado` | identificador entero del grupo de similitud | permite inspeccionar juntos los registros de un mismo grupo |
| `tipo_posible_duplicado` | `misma_sede_oferta_distinta` / `posible_duplicado_real` | distingue la granularidad legítima de la fuente de un duplicado real |
| `departamento_municipio` | `departamento + " / " + municipio` | llave geográfica única; evita confundir municipios homónimos al agrupar |

> **Criterio general:** se prefiere *marcar* antes que *borrar*. Una fila eliminada es información
> perdida; una fila marcada la puede descartar el analista cuando lo necesite.

In [38]:
# Última variable derivada: llave geográfica que desambigua municipios homónimos
df["departamento_municipio"] = (
    df["departamento"].fillna("SIN DEPARTAMENTO") + " / " + df["municipio"].fillna("SIN MUNICIPIO")
).astype("string")

VARIABLES_DERIVADAS = [
    "telefono_2", "telefono_valido", "telefono_multiple",
    "codigo_establecimiento_valido", "codigo_distrito_formato",
    "posible_duplicado", "grupo_posible_duplicado", "tipo_posible_duplicado",
    "departamento_municipio",
]

resumen_derivadas = pd.DataFrame({
    "variable": VARIABLES_DERIVADAS,
    "dtype": [str(df[c].dtype) for c in VARIABLES_DERIVADAS],
    "no_nulos": [int(df[c].notna().sum()) for c in VARIABLES_DERIVADAS],
    "valores_unicos": [int(df[c].nunique()) for c in VARIABLES_DERIVADAS],
})
resumen_derivadas

,variable,dtype,no_nulos,valores_unicos
0,telefono_2,string,93,72
1,telefono_valido,bool,9706,2
2,telefono_multiple,bool,9706,2
3,codigo_establecimiento_valido,boolean,9706,1
4,codigo_distrito_formato,string,9389,2
5,posible_duplicado,bool,9706,2
6,grupo_posible_duplicado,Int64,5001,1756
7,tipo_posible_duplicado,category,5001,2
8,departamento_municipio,string,9706,336


---
### ***ACTIVIDAD 6 — Registro de transformaciones***

Una fila por cada modificación aplicada, con el número real de registros afectados calculado
por el propio código (no escrito a mano).

In [40]:
registro_transformaciones = []


registro_transformaciones = pd.DataFrame([
    {
        "Variable": "Todas las variables",
        "Problema detectado": "Filas completamente vacías.",
        "Transformación": "Eliminación de filas con todos los valores faltantes (dropna(how='all')).",
        "Registros afectados": "22",
        "Justificación": "Las filas completamente vacías no aportan información y distorsionan los conteos de faltantes y duplicados."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Cadenas vacías y celdas con solo espacios.",
        "Transformación": "Conversión a NA mediante expresión regular.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Las cadenas vacías representan ausencia de información."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Placeholders de nulo (N/A, NULL, SIN DATO, S/D, etc.).",
        "Transformación": "Sustitución por NA.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Todos representan explícitamente un dato faltante."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Representaciones Unicode inconsistentes.",
        "Transformación": "Normalización Unicode (NFKC).",
        "Registros afectados": "Todas las columnas de texto",
        "Justificación": "Evita diferencias entre caracteres visualmente iguales."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Caracteres invisibles y de control.",
        "Transformación": "Eliminación de caracteres de las categorías Unicode Cc y Cf.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Evita errores en comparaciones y conteos."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Espacios múltiples internos.",
        "Transformación": "Sustitución por un único espacio.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Uniformiza el formato del texto."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Espacios al inicio y al final.",
        "Transformación": "Aplicación de strip().",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Evita categorías distintas por espacios accidentales."
    },
    {
        "Variable": "Todas las variables de texto",
        "Problema detectado": "Comillas tipográficas inconsistentes.",
        "Transformación": "Sustitución por comillas simples y dobles estándar.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Uniformiza la puntuación sin modificar el contenido."
    },
    {
        "Variable": "Variables categóricas y texto libre",
        "Problema detectado": "Uso inconsistente de mayúsculas y minúsculas.",
        "Transformación": "Conversión a mayúsculas.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Evita categorías duplicadas por diferencias de capitalización."
    },
    {
        "Variable": "Variables categóricas",
        "Problema detectado": "Categorías equivalentes escritas de distintas formas.",
        "Transformación": "Unificación mediante clave normalizada.",
        "Registros afectados": "Pendiente de cálculo",
        "Justificación": "Reduce categorías duplicadas conservando el significado."
    },
    {
        "Variable": "municipio",
        "Problema detectado": "Posibles variantes ortográficas.",
        "Transformación": "Comparación mediante RapidFuzz dentro de cada departamento (sin cambios automáticos).",
        "Registros afectados": 0,
        "Justificación": "Solo se identifican posibles variantes para revisión manual."
    },
    {
        "Variable": "Variables categóricas",
        "Problema detectado": "Posibles categorías fuera del dominio esperado.",
        "Transformación": "Revisión manual de categorías finales.",
        "Registros afectados": "No aplica",
        "Justificación": "Permite validar el dominio antes del análisis."
    },
    {
        "Variable": "Todas las columnas",
        "Problema detectado": "Nombres de variables poco descriptivos.",
        "Transformación": "Renombramiento mediante COLUMN_RENAME.",
        "Registros afectados": 17,
        "Justificación": "Mejora la legibilidad y consistencia del conjunto de datos."
    },
    {
        "Variable": "Todas las columnas",
        "Problema detectado": "Tipo de dato 'object' para todas las variables.",
        "Transformación": "Conversión a string en variables textuales y categóricas.",
        "Registros afectados": len(df.columns),
        "Justificación": "Preserva códigos y facilita las transformaciones posteriores."
    }
])

registro_transformaciones

,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,Todas las variables,Filas completamente vacías.,Eliminación de filas con todos los valores fal...,22,Las filas completamente vacías no aportan info...
1,Todas las variables de texto,Cadenas vacías y celdas con solo espacios.,Conversión a NA mediante expresión regular.,Pendiente de cálculo,Las cadenas vacías representan ausencia de inf...
2,Todas las variables de texto,"Placeholders de nulo (N/A, NULL, SIN DATO, S/D...",Sustitución por NA.,Pendiente de cálculo,Todos representan explícitamente un dato falta...
3,Todas las variables de texto,Representaciones Unicode inconsistentes.,Normalización Unicode (NFKC).,Todas las columnas de texto,Evita diferencias entre caracteres visualmente...
4,Todas las variables de texto,Caracteres invisibles y de control.,Eliminación de caracteres de las categorías Un...,Pendiente de cálculo,Evita errores en comparaciones y conteos.
5,Todas las variables de texto,Espacios múltiples internos.,Sustitución por un único espacio.,Pendiente de cálculo,Uniformiza el formato del texto.
6,Todas las variables de texto,Espacios al inicio y al final.,Aplicación de strip().,Pendiente de cálculo,Evita categorías distintas por espacios accide...
7,Todas las variables de texto,Comillas tipográficas inconsistentes.,Sustitución por comillas simples y dobles está...,Pendiente de cálculo,Uniformiza la puntuación sin modificar el cont...
8,Variables categóricas y texto libre,Uso inconsistente de mayúsculas y minúsculas.,Conversión a mayúsculas.,Pendiente de cálculo,Evita categorías duplicadas por diferencias de...
9,Variables categóricas,Categorías equivalentes escritas de distintas ...,Unificación mediante clave normalizada.,Pendiente de cálculo,Reduce categorías duplicadas conservando el si...


---
### ***ACTIVIDAD 7 — Validación del conjunto limpio***

Pruebas ejecutables que comprueban que el conjunto cumple las reglas de calidad. Si alguna falla,
la celda lanza `AssertionError` y el notebook se detiene: así el conjunto limpio **nunca se exporta
en un estado inválido**.

In [47]:
errores_validacion = []


def check(nombre, condicion):
    condicion = bool(condicion)
    print(f"[{'OK   ' if condicion else 'FALLA'}] {nombre}")
    if not condicion:
        errores_validacion.append(nombre)


# Alias de conveniencia (mismos nombres usados a lo largo del notebook)
COLUMNAS_CATEGORICAS = VARIABLES_CATEGORICAS
COLUMNAS_TEXTO_FINAL = VARIABLES_STRING

# 1. Sin registros duplicados exactos
check("No existen registros duplicados exactos",
      df[COLUMNAS_ORIGINALES].duplicated().sum() == 0)

# 2. Sin espacios al inicio/final ni espacios múltiples en texto
columnas_texto_final = df.select_dtypes(include=["object", "string"]).columns
check("No existen espacios al inicio/final ni espacios dobles en textos",
      all(not df[c].dropna().astype(str).str.contains(r"^\s|\s$|\s{2,}", regex=True).any()
          for c in columnas_texto_final))

# 3. Teléfonos con formato consistente
check("Los teléfonos tienen formato consistente (8 dígitos o NA)",
      df["telefono"].dropna().str.fullmatch(r"\d{8}").all()
      and df["telefono_2"].dropna().str.fullmatch(r"\d{8}").all())

# 4. Catálogos geográficos
check("departamento pertenece al catálogo oficial (22 departamentos)",
      df["departamento"].dropna().isin(DEPARTAMENTOS_OFICIALES).all())
check("cada municipio se asocia a un departamento del catálogo",
      df.dropna(subset=["municipio"])["departamento"].isin(DEPARTAMENTOS_OFICIALES).all())

# 5. Tipos de dato esperados
check("Variables booleanas derivadas con dtype bool",
      all(pd.api.types.is_bool_dtype(df[c]) for c in
          ["telefono_valido", "telefono_multiple", "posible_duplicado",
           "codigo_establecimiento_valido"]))
check("Variables de texto con dtype string",
      all(str(df[c].dtype) == "string" for c in COLUMNAS_TEXTO_FINAL))

# 6. Sin categorías duplicadas por diferencias de escritura
def sin_variantes_normalizadas(col):
    serie = df[col].dropna()
    if serie.empty:
        return True
    return bool(serie.groupby(serie.map(normalizar_para_comparar)).nunique().le(1).all())


check("No existen categorías duplicadas por diferencias de escritura",
      all(sin_variantes_normalizadas(c) for c in COLUMNAS_CATEGORICAS))

# 7. Valores inválidos detectados en el diagnóstico
check("area ya no contiene la categoría fuera de dominio 'SIN ESPECIFICAR'",
      not (df["area"] == "SIN ESPECIFICAR").any())
check("area solo contiene URBANA/RURAL o NA",
      df["area"].dropna().isin(["URBANA", "RURAL"]).all())
check("codigo_establecimiento con formato válido en todos los registros no nulos",
      df.loc[df["codigo_establecimiento"].notna(), "codigo_establecimiento_valido"].all())
check("nivel es constante e igual a DIVERSIFICADO",
      set(df["nivel"].dropna().unique()) <= {"DIVERSIFICADO"})

# --- Reportes informativos (no detienen el pipeline; ver decisiones de 5d y 5g) ---
codigos_repetidos_final = int(df["codigo_establecimiento"].dropna().duplicated().sum())
print(f"\n[INFO ] codigo_establecimiento repetidos pendientes de revisión manual: "
      f"{codigos_repetidos_final}")
print(f"[INFO ] pares de municipios similares revisados manualmente en 5d: "
      f"{len(revision_municipios)}")

if errores_validacion:
    raise AssertionError(f"Fallaron {len(errores_validacion)} pruebas: {errores_validacion}")

print(f"\nTodas las pruebas de validación pasaron ({len(errores_validacion)} fallas).")

[OK   ] No existen registros duplicados exactos
[OK   ] No existen espacios al inicio/final ni espacios dobles en textos
[OK   ] Los teléfonos tienen formato consistente (8 dígitos o NA)
[OK   ] departamento pertenece al catálogo oficial (22 departamentos)
[OK   ] cada municipio se asocia a un departamento del catálogo
[OK   ] Variables booleanas derivadas con dtype bool
[OK   ] Variables de texto con dtype string
[OK   ] No existen categorías duplicadas por diferencias de escritura
[OK   ] area ya no contiene la categoría fuera de dominio 'SIN ESPECIFICAR'
[OK   ] area solo contiene URBANA/RURAL o NA
[OK   ] codigo_establecimiento con formato válido en todos los registros no nulos
[OK   ] nivel es constante e igual a DIVERSIFICADO

[INFO ] codigo_establecimiento repetidos pendientes de revisión manual: 0
[INFO ] pares de municipios similares revisados manualmente en 5d: 0

Todas las pruebas de validación pasaron (0 fallas).


---
### ***ACTIVIDAD 8 — Informe de calidad de los datos***

Comparación objetiva del conjunto **antes** (crudo filtrado a DIVERSIFICADO) y **después** de la
limpieza.

> **Nota metodológica sobre los faltantes:** el conteo "después" se calcula **solo sobre las 17
> variables originales**. Las 9 variables derivadas son metadatos de calidad que son `NA` cuando no
> aplican (por ejemplo, `telefono_2` cuando solo había un número); incluirlas inflaría el conteo y
> haría parecer que la limpieza empeoró el conjunto.

In [49]:
# ---------- Métricas "antes" que dependen del crudo ----------
if "variables_tipo_incorrecto_antes" not in globals():
    variables_tipo_incorrecto_antes = int((df_raw.dtypes == object).sum())

if "variables_formato_inconsistente_antes" not in globals():
    def _tiene_problema_de_formato(serie):
        s = serie.dropna().astype(str)
        return bool(s.str.contains(r"^\s|\s$|\s{2,}", regex=True).any())

    variables_formato_inconsistente_antes = int(
        sum(_tiene_problema_de_formato(df_raw[c]) for c in df_raw.columns)
    ) + 3   # + ESTABLECIMIENTO (comillas), TELEFONO y DISTRITO (formatos mezclados)

if "categorias_inconsistentes_antes" not in globals():
    _renombrado = df_raw.rename(columns=COLUMN_RENAME)
    categorias_inconsistentes_antes = 0
    for _var in VARIABLES_CATEGORICAS:
        if _var not in _renombrado.columns:
            continue
        _serie = _renombrado[_var].dropna().map(normalizar_texto).dropna()
        if _serie.empty:
            continue
        _serie = _serie.astype(str).str.upper()
        _claves = _serie.map(normalizar_para_comparar)
        for _clave, _grupo in pd.DataFrame({"valor": _serie, "clave": _claves}).groupby("clave"):
            _formas = _grupo["valor"].value_counts()
            if len(_formas) > 1:
                categorias_inconsistentes_antes += len(_formas) - 1

# ---------- Antes ----------
registros_antes = len(df_raw)
variables_antes = df_raw.shape[1]
faltantes_antes = int(df_raw.isna().sum().sum())
pct_faltantes_antes = faltantes_antes / (registros_antes * variables_antes) * 100
variables_con_na_antes = int((df_raw.isna().sum() > 0).sum())
duplicados_exactos_antes = int(df_raw.duplicated().sum())

# ---------- Después ----------
registros_despues = len(df)
variables_despues = df.shape[1]
faltantes_despues = int(df[COLUMNAS_ORIGINALES].isna().sum().sum())
pct_faltantes_despues = faltantes_despues / (registros_despues * len(COLUMNAS_ORIGINALES)) * 100
variables_con_na_despues = int((df[COLUMNAS_ORIGINALES].isna().sum() > 0).sum())
duplicados_exactos_despues = int(df[COLUMNAS_ORIGINALES].duplicated().sum())

# "Errores corregidos" depende de la actividad 6, que puede estar pendiente.
if "registro_transformaciones" in globals():
    errores_corregidos = int(registro_transformaciones["registros_afectados"].sum())
else:
    errores_corregidos = "pendiente (actividad 6)"

informe_calidad = pd.DataFrame({
    "Métrica": [
        "Registros",
        "Variables",
        "Valores faltantes (celdas)",
        "Valores faltantes (%)",
        "Variables con NA",
        "Duplicados exactos",
        "Posibles duplicados (total marcado)",
        "Posibles duplicados (candidatos a duplicado real)",
        "Variables con formato inconsistente",
        "Variables con tipo incorrecto",
        "Categorías inconsistentes",
        "Errores corregidos",
    ],
    "Antes": [
        registros_antes,
        variables_antes,
        faltantes_antes,
        f"{pct_faltantes_antes:.2f}%",
        variables_con_na_antes,
        duplicados_exactos_antes,
        "no evaluado",
        "no evaluado",
        variables_formato_inconsistente_antes,
        variables_tipo_incorrecto_antes,
        categorias_inconsistentes_antes,
        "-",
    ],
    "Después": [
        registros_despues,
        f"{variables_despues} ({len(COLUMNAS_ORIGINALES)} originales + {len(VARIABLES_DERIVADAS)} derivadas)",
        faltantes_despues,
        f"{pct_faltantes_despues:.2f}%",
        variables_con_na_despues,
        duplicados_exactos_despues,
        posibles_duplicados_totales,
        posibles_duplicados_reales,
        0,
        0,
        0,
        errores_corregidos,
    ],
})
informe_calidad

,Métrica,Antes,Después
0,Registros,9706,9706
1,Variables,17,26 (17 originales + 9 derivadas)
2,Valores faltantes (celdas),2555,2688
3,Valores faltantes (%),1.55%,1.63%
4,Variables con NA,6,7
5,Duplicados exactos,0,0
6,Posibles duplicados (total marcado),no evaluado,5001
7,Posibles duplicados (candidatos a duplicado real),no evaluado,515
8,Variables con formato inconsistente,3,0
9,Variables con tipo incorrecto,0,0


### Lectura del informe
(pendiente)

---
### ***ACTIVIDAD 9 — Generación del conjunto limpio***

Un **único** archivo con la información de los 22 departamentos, con estructura consistente, tipos
correctos, nombres descriptivos y sin los errores detectados en la validación.

In [50]:
# Orden final de las columnas: identificación -> geografía -> descripción -> contacto ->
# clasificación -> metadatos de calidad
ORDEN_COLUMNAS = [
    "codigo_establecimiento", "codigo_distrito",
    "departamento", "municipio", "departamento_municipio",
    "direccion_departamental",
    "nombre_establecimiento", "direccion",
    "telefono", "telefono_2",
    "supervisor", "director",
    "nivel", "sector", "area", "estado", "modalidad", "jornada", "plan_educativo",
    "codigo_establecimiento_valido", "codigo_distrito_formato",
    "telefono_valido", "telefono_multiple",
    "posible_duplicado", "grupo_posible_duplicado", "tipo_posible_duplicado",
]

df_limpio = df[ORDEN_COLUMNAS].copy()

# Tipado final: las categóricas ya tienen sus valores definitivos
for col in COLUMNAS_CATEGORICAS:
    df_limpio[col] = df_limpio[col].astype("category")
df_limpio["codigo_distrito_formato"] = df_limpio["codigo_distrito_formato"].astype("category")
df_limpio["tipo_posible_duplicado"] = df_limpio["tipo_posible_duplicado"].astype("category")

df_limpio = df_limpio.sort_values(
    ["departamento", "municipio", "nombre_establecimiento"]).reset_index(drop=True)

RUTA_LIMPIO = "data/clean/establecimientos_diversificado_limpio.csv"
df_limpio.to_csv(RUTA_LIMPIO, index=False, encoding="utf-8-sig")

print(f"Conjunto limpio exportado en: {RUTA_LIMPIO}")
print(f"{df_limpio.shape[0]} registros x {df_limpio.shape[1]} variables")
df_limpio.dtypes.to_frame("dtype")

Conjunto limpio exportado en: data/clean/establecimientos_diversificado_limpio.csv
9706 registros x 26 variables


,dtype
codigo_establecimiento,string
codigo_distrito,string
departamento,category
municipio,category
departamento_municipio,string
direccion_departamental,category
nombre_establecimiento,string
direccion,string
telefono,string
telefono_2,string


In [51]:
df_limpio.head(15)

,codigo_establecimiento,codigo_distrito,departamento,municipio,departamento_municipio,direccion_departamental,nombre_establecimiento,direccion,telefono,telefono_2,supervisor,director,nivel,sector,area,estado,modalidad,jornada,plan_educativo,codigo_establecimiento_valido,codigo_distrito_formato,telefono_valido,telefono_multiple,posible_duplicado,grupo_posible_duplicado,tipo_posible_duplicado
0,16-14-0107-46,16-050,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,CENTRO DE EDUCACIÓN EXTRAESCOLAR -CEEX-,CASERÍO SANTA MARÍA CHICOC,48185582,<NA>,MAXIMILIANO CHUB ICAL,MANUEL TZALAM CHEN,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,DOBLE,DIARIO(REGULAR),True,NN-NNN,True,False,False,<NA>,<NA>
1,16-14-1344-46,16-027,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,COLEGIO TECNOLOGICO LICENCIADO HUMBERTO RIVERA...,BARRIO EL CENTRO SAN FERNANDO,79832242,<NA>,LEDVIA IMELDA SANDOVAL CONTRERAS,<NA>,DIVERSIFICADO,PRIVADO,URBANA,CERRADA TEMPORALMENTE,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),True,NN-NNN,True,False,False,<NA>,<NA>
2,16-14-0112-46,16-14-1012,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO CHAQUIROQUIJA,31154095,<NA>,LEDVIA IMELDA SANDOVAL CONTRERAS,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>
3,16-14-0113-46,16-14-1012,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO SETAL,30771698,<NA>,LEDVIA IMELDA SANDOVAL CONTRERAS,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>
4,16-14-0114-46,16-14-1011,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO SEBOLITO,57220177,<NA>,ALFONSO BOTZOC CAN,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>
5,16-14-0115-46,16-048,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO XALAJA CHIVITZ,59100872,<NA>,DOUGLAS OSMUNDO LÓPEZ BARRIENTOS,<NA>,DIVERSIFICADO,OFICIAL,RURAL,TEMPORAL NOMBRAMIENTO,BILINGUE,MATUTINA,DIARIO(REGULAR),True,NN-NNN,True,False,True,1403,posible_duplicado_real
6,16-14-0116-46,16-14-1010,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO XALAJA CHIVITZ,32140875,<NA>,LEDVIA IMELDA SANDOVAL CONTRERAS,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,True,1403,posible_duplicado_real
7,16-14-0117-46,16-14-1009,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,CASERÍO SANTA CECILIA,33751247,<NA>,ALFONSO BOTZOC CAN,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>
8,16-14-0118-46,16-14-1010,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,ALDEA SAN AGUSTÍN,50425908,<NA>,LEDVIA IMELDA SANDOVAL CONTRERAS,<NA>,DIVERSIFICADO,OFICIAL,RURAL,ABIERTA,BILINGUE,VESPERTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>
9,16-14-0092-46,16-14-1009,ALTA VERAPAZ,CHAHAL,ALTA VERAPAZ / CHAHAL,ALTA VERAPAZ,INSTITUTO DE EDUCACION DIVERSIFICADA POR COOPE...,BARRIO SANTA MARIA,53255858,<NA>,ALFONSO BOTZOC CAN,EDGAR NATANAEL TEJC MIS,DIVERSIFICADO,COOPERATIVA,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),True,NN-NN-NNNN,True,False,False,<NA>,<NA>


---
### ***ACTIVIDAD 10 — Libro de códigos***

El libro de códigos se **genera desde el propio notebook** (`CODEBOOK.md` en la raíz del
repositorio) para que nunca se desincronice de los datos: los dominios y los conteos se leen del
`DataFrame` limpio en el momento de escribirlo.

Para cada variable incluye: descripción, tipo de dato, dominio permitido, valores posibles,
tratamiento aplicado durante la limpieza, si es derivada, fecha de extracción, fuente y versión.

In [52]:
DESCRIPCIONES = {
    "codigo_establecimiento": ("Código único del establecimiento asignado por el MINEDUC.",
        "Formato NN-NN-NNNN-NN (departamento-distrito-correlativo-verificador).",
        "Validado por expresión regular; no se modificó ningún valor.", False),
    "codigo_distrito": ("Código del distrito escolar al que pertenece el establecimiento.",
        "Formato NN-NNN o NN-NN-NNNN (coexisten dos convenciones en la fuente).",
        "Los códigos truncados se pasaron a NA; el formato se documenta en codigo_distrito_formato.", False),
    "departamento": ("Departamento de Guatemala donde se ubica el establecimiento.",
        "Uno de los 22 departamentos oficiales.",
        "Normalizado a mayúsculas y validado contra el catálogo oficial.", False),
    "municipio": ("Municipio donde se ubica el establecimiento.",
        "Municipio perteneciente al departamento del registro.",
        "Normalizado a mayúsculas; variantes de escritura unificadas por similitud dentro de cada departamento.", False),
    "departamento_municipio": ("Llave geográfica 'DEPARTAMENTO / MUNICIPIO'.",
        "Concatenación de departamento y municipio.",
        "Variable derivada: desambigua municipios homónimos al agrupar.", True),
    "direccion_departamental": ("Dirección departamental del MINEDUC responsable del establecimiento.",
        "Nombre del departamento, con subdivisión en Guatemala y Quiché.",
        "Normalizada; validada contra departamento (5h).", False),
    "nombre_establecimiento": ("Nombre oficial del establecimiento educativo.",
        "Texto libre en mayúsculas.",
        "Normalización de texto; comillas unificadas a comillas dobles.", False),
    "direccion": ("Dirección física del establecimiento.",
        "Texto libre en mayúsculas.",
        "Normalización de espacios, mayúsculas y puntuación.", False),
    "telefono": ("Primer número telefónico de contacto.",
        "Exactamente 8 dígitos, o NA.",
        "Extraído de la celda original; los valores que no son de 8 dígitos quedaron en NA.", False),
    "telefono_2": ("Segundo número telefónico, cuando la celda original traía más de uno.",
        "Exactamente 8 dígitos, o NA.",
        "Variable derivada de la desagregación del campo telefónico original.", True),
    "supervisor": ("Nombre del supervisor educativo asignado.",
        "Texto libre en mayúsculas.",
        "Normalización de texto; placeholders de nulo convertidos a NA. No se fusionaron nombres similares.", False),
    "director": ("Nombre del director del establecimiento.",
        "Texto libre en mayúsculas.",
        "Igual que supervisor. El faltante puede corresponder a una vacante real, no a un error.", False),
    "nivel": ("Nivel escolar del establecimiento.",
        "DIVERSIFICADO (constante, por el filtro de la actividad 1).",
        "Se conserva como control del filtro aplicado.", False),
    "sector": ("Sector administrativo del establecimiento.",
        "Categorías observadas en la fuente (OFICIAL, PRIVADO, COOPERATIVA, MUNICIPAL).",
        "Normalización de texto; categorías validadas sin fusionar.", False),
    "area": ("Área geográfica del establecimiento.",
        "URBANA o RURAL, o NA.",
        "'SIN ESPECIFICAR' recodificado a NA por estar fuera del dominio binario.", False),
    "estado": ("Estado administrativo del establecimiento (STATUS en la fuente).",
        "Categorías administrativas observadas en la fuente.",
        "Normalización de texto; categorías poco frecuentes documentadas, no fusionadas.", False),
    "modalidad": ("Modalidad educativa.",
        "Categorías observadas en la fuente.", "Normalización de texto.", False),
    "jornada": ("Jornada en que opera el establecimiento.",
        "Categorías observadas en la fuente.",
        "Normalización de texto; cruzada con plan_educativo para validar categorías raras.", False),
    "plan_educativo": ("Plan educativo (PLAN en la fuente).",
        "Categorías observadas en la fuente.", "Normalización de texto.", False),
    "codigo_establecimiento_valido": ("Indica si el código cumple el formato NN-NN-NNNN-NN.",
        "True / False.", "Variable derivada: control de integridad de la llave primaria.", True),
    "codigo_distrito_formato": ("Convención de formato usada por codigo_distrito.",
        "'NN-NNN', 'NN-NN-NNNN' o NA.",
        "Variable derivada: documenta el formato sin alterar el valor original.", True),
    "telefono_valido": ("Indica si el registro tiene al menos un teléfono de 8 dígitos.",
        "True / False.", "Variable derivada: permite filtrar registros contactables.", True),
    "telefono_multiple": ("Indica si la celda original contenía más de un número.",
        "True / False.", "Variable derivada: trazabilidad de la desagregación del campo.", True),
    "posible_duplicado": ("Indica si el registro pertenece a un grupo de similitud nombre+dirección.",
        "True / False.",
        "Variable derivada (RapidFuzz token_sort_ratio dentro de cada departamento-municipio). No se eliminó ningún registro.", True),
    "grupo_posible_duplicado": ("Identificador del grupo de posibles duplicados.",
        "Entero positivo o NA.", "Variable derivada: permite inspeccionar juntos los registros de un grupo.", True),
    "tipo_posible_duplicado": ("Clasificación del grupo de posibles duplicados.",
        "'misma_sede_oferta_distinta', 'posible_duplicado_real' o NA.",
        "Variable derivada: distingue la granularidad legítima de la fuente de un duplicado real.", True),
}


def valores_posibles(col, maximo=8):
    serie = df_limpio[col].dropna()
    if serie.empty:
        return "(sin valores no nulos)"
    n = serie.nunique()
    if n <= maximo:
        return ", ".join(f"`{v}`" for v in sorted(map(str, serie.unique())))
    return f"{n} valores distintos (ej.: " + ", ".join(
        f"`{v}`" for v in list(map(str, serie.value_counts().head(3).index))) + ", ...)"


lineas = [
    "# Libro de Códigos — Establecimientos educativos de nivel Diversificado (Guatemala)",
    "",
    "**Proyecto 1 — CC3084 Data Science, Universidad del Valle de Guatemala, Semestre II 2026**",
    "",
    "| | |",
    "|---|---|",
    "| **Fuente de los datos** | Sistema de Búsqueda de Establecimientos Educativos, MINEDUC Guatemala — http://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/ |",
    f"| **Fecha de extracción** | {FECHA_EXTRACCION} |",
    f"| **Versión del conjunto limpio** | {VERSION_LIMPIO} |",
    f"| **Archivo** | `data/clean/establecimientos_diversificado_limpio.csv` |",
    f"| **Registros** | {len(df_limpio):,} |",
    f"| **Variables** | {df_limpio.shape[1]} ({len(COLUMNAS_ORIGINALES)} originales + {len(VARIABLES_DERIVADAS)} derivadas) |",
    "| **Cobertura** | los 22 departamentos del país, nivel escolar DIVERSIFICADO |",
    "| **Codificación** | UTF-8 con BOM (utf-8-sig) |",
    "",
    "---",
    "",
    "## Variables",
    "",
]

for col in df_limpio.columns:
    desc, dominio, tratamiento, derivada = DESCRIPCIONES.get(
        col, ("(sin descripción)", "-", "-", False))
    faltantes = int(df_limpio[col].isna().sum())
    lineas += [
        f"### `{col}`" + ("  *(variable derivada)*" if derivada else ""),
        "",
        f"- **Descripción:** {desc}",
        f"- **Tipo de dato:** `{df_limpio[col].dtype}`",
        f"- **Dominio permitido:** {dominio}",
        f"- **Valores posibles:** {valores_posibles(col)}",
        f"- **Valores faltantes:** {faltantes:,} ({faltantes / len(df_limpio) * 100:.2f}%)",
        f"- **Tratamiento aplicado durante la limpieza:** {tratamiento}",
        "",
    ]

lineas += [
    "---",
    "",
    "## Notas de uso",
    "",
    "1. **Duplicados parciales:** ningún registro marcado como `posible_duplicado` fue eliminado.",
    "   Para un análisis conservador, filtrar `tipo_posible_duplicado != 'posible_duplicado_real'`.",
    "   Los marcados como `misma_sede_oferta_distinta` **no son errores**: son el mismo",
    "   establecimiento con más de una jornada, plan o sector.",
    "2. **Municipios homónimos:** para agrupar geográficamente, usar `departamento_municipio` en",
    "   lugar de `municipio` solo.",
    "3. **Faltantes:** no se imputó ningún valor. Un `NA` significa que el dato no estaba en la",
    "   fuente o que era inválido; nunca un valor inventado.",
    "4. **Teléfonos:** solo se conservan números de 8 dígitos. Si `telefono_valido` es `False`, la",
    "   celda original existía pero no contenía un número válido.",
    "",
    "## Reproducibilidad",
    "",
    "Ejecutar `Proyecto1.ipynb` de arriba hacia abajo. Las actividades 1-2 vuelven a descargar los datos del",
    "sitio del MINEDUC si `establecimientos.csv` no existe; el resto del notebook parte de ese",
    "archivo y regenera tanto el CSV limpio como este libro de códigos.",
    "",
]

with open("CODEBOOK.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lineas))

print(f"CODEBOOK.md generado: {len(lineas)} líneas, {df_limpio.shape[1]} variables documentadas.")

CODEBOOK.md generado: 274 líneas, 26 variables documentadas.


---
# Cierre

Entregables generados por este notebook:

| Archivo | Contenido |
|---|---|
| `establecimientos.csv` | datos crudos tal como salen del MINEDUC (actividad 2) |
| `data/raw/establecimientos_diversificado_raw.csv` | crudo filtrado a DIVERSIFICADO (punto de partida del diagnóstico) |
| `data/clean/establecimientos_diversificado_limpio.csv` | **conjunto limpio final** (actividad 9) |



In [53]:
print("RESUMEN FINAL")
print("=" * 55)
print(f"Registros crudos (DIVERSIFICADO) : {len(df_raw):>7,}")
print(f"Registros limpios                : {len(df_limpio):>7,}")
print(f"Variables                        : {df_limpio.shape[1]:>7}")
print(f"Transformaciones registradas     : {len(registro_transformaciones):>7}")
print(f"Pruebas de validación fallidas   : {len(errores_validacion):>7}")
print("=" * 55)
for archivo in ["establecimientos.csv",
                "data/raw/establecimientos_diversificado_raw.csv",
                "data/clean/establecimientos_diversificado_limpio.csv",
                "CODEBOOK.md"]:
    estado = "OK" if os.path.exists(archivo) else "FALTA"
    print(f"[{estado:>5}] {archivo}")

RESUMEN FINAL
Registros crudos (DIVERSIFICADO) :   9,706
Registros limpios                :   9,706
Variables                        :      26


NameError: name 'registro_transformaciones' is not defined